# **Knowledge Graph 구현**

---

## 1. Neo4J Desktop 환경 설정

- 버전 1.6.2 설치: https://neo4j.com/deployment-center/

- Neo4J Desktop 설치 후, Neo4J Desktop에서 새로운 프로젝트 생성
    - 프로젝트 이름: `test`
    - 데이터베이스 버전: `5.24.0`

- 데이터베이스에서 플러그인 설치
    - `APOC`

- 데이터베이스 Settings에서 다음 설정 추가 (apoc 검색해서 기존 설정에 추가)
    - `dbms.security.procedures.unrestricted=jwt.security.*,apoc.*,apoc.meta.*`

- Neo4J Desktop 사용(.env)
    ```
    NEO4J_URI=bolt://localhost:7687
    NEO4J_USERNAME=neo4j
    NEO4J_PASSWORD=modulab1234
    NEO4J_DATABASE=neo4j
    ```

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

`(3) Neo4j 설정`

In [3]:
import os
from langchain_neo4j import Neo4jGraph

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")


# LangChain 도구 활용 - DB 연결 객체 초기화 
graph = Neo4jGraph(  
    url=NEO4J_URI, 
    username=NEO4J_USERNAME, 
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    )

graph.query("MATCH (n) RETURN n LIMIT 5;")

[{'n': {'trackingError': 0.11,
   'netAsset': 111916276404,
   'replicationMethod': '실물(액티브)',
   'listingDate': '2023/09/19',
   'code': '466400',
   'baseIndex': 'KIS 2025-08만기형 크레딧 A+이상 지수(총수익)',
   'volatility': '매우낮음',
   'yearReturn': 4.52,
   'disparityRatio': 0.03,
   'totalFee': 0.1,
   'name': '1Q 25-08 회사채(A+이상)액티브',
   'company': '하나자산운용',
   'embedding': [0.013379422947764397,
    0.017141688615083694,
    -0.007986469194293022,
    0.016262343153357506,
    0.043655652552843094,
    -0.01364656537771225,
    -0.05284983292222023,
    0.04305458068847656,
    0.02713729813694954,
    -0.06625151634216309,
    0.02230645902454853,
    0.0019715726375579834,
    -0.030454326421022415,
    -0.008320397697389126,
    0.03317028284072876,
    -0.014537042938172817,
    -0.0211822297424078,
    -0.057747457176446915,
    -0.0743548572063446,
    -0.0751117691397667,
    0.01107531227171421,
    -0.04016052931547165,
    -0.009116262197494507,
    0.018911512568593025,
    -0.024

`(4) 기존 DB의 모든 내용 삭제`

In [4]:
def reset_database(graph):
    """
    데이터베이스 초기화하기
    """
    # 모든 노드와 관계 삭제
    graph.query("MATCH (n) DETACH DELETE n")
    
    # 모든 제약조건 삭제
    constraints = graph.query("SHOW CONSTRAINTS")
    for constraint in constraints:
        constraint_name = constraint.get("name")
        if constraint_name:
            graph.query(f"DROP CONSTRAINT {constraint_name}")
    
    # 모든 인덱스 삭제
    indexes = graph.query("SHOW INDEXES")
    for index in indexes:
        index_name = index.get("name")
        index_type = index.get("type")
        if index_name and index_type != "CONSTRAINT":
            graph.query(f"DROP INDEX {index_name}")
    
    print("데이터베이스가 초기화되었습니다.")

# 데이터베이스 초기화
reset_database(graph)

데이터베이스가 초기화되었습니다.


In [5]:
# 그래프 스키마 조회
graph.refresh_schema()
print(graph.schema)

Node properties:

Relationship properties:

The relationships:



## 2. 정형 데이터를 KG로 변환

`(1) Load CSV Data`

- etf info 데이터 활용 (data/etf_list.csv)

In [6]:
# CSV 파일 읽기
df = pd.read_csv('data/etf_list.csv', encoding='cp949')

df.shape

(930, 14)

In [7]:
df.head()

,종목코드,종목명,상장일,분류체계,운용사,수익률(최근 1년),기초지수,추적오차,순자산총액,괴리율,변동성,복제방법,총보수,과세유형
0,466400,1Q 25-08 회사채(A+이상)액티브,2023/09/19,채권-회사채-단기,하나자산운용,4.52,KIS 2025-08만기형 크레딧 A+이상 지수(총수익),0.11,111916276404,0.03,매우낮음,실물(액티브),0.10,배당소득세(보유기간과세)
1,491610,1Q CD금리액티브(합성),2024/09/24,기타,하나자산운용,0.00,KIS 하나 CD금리 총수익지수,0.05,316206006696,0.02,매우낮음,합성(액티브),0.02,배당소득세(보유기간과세)
2,451060,1Q K200액티브,2023/01/31,주식-시장대표,하나자산운용,-3.66,코스피 200,0.77,99754348820,-0.01,높음,실물(액티브),0.18,배당소득세(보유기간과세)
3,463290,1Q 단기금융채액티브,2023/08/03,채권-혼합-단기,하나자산운용,4.01,MK 머니마켓 지수(총수익),0.05,252717462257,0.00,매우낮음,실물(액티브),0.08,배당소득세(보유기간과세)
4,479080,1Q 머니마켓액티브,2024/04/02,채권-혼합-단기,하나자산운용,0.00,KIS-하나 MMF 지수(총수익),0.06,308255065986,-0.01,매우낮음,실물(액티브),0.05,배당소득세(보유기간과세)


`(2) 제약 조건 생성`

In [8]:
# 종목코드(code)는 ETF마다 고유하므로 유일성 제약조건을 설정 
unique_code_constraint = """
CREATE CONSTRAINT etf_code_unique IF NOT EXISTS
FOR (e:ETF) 
REQUIRE e.code IS UNIQUE
"""
graph.query(unique_code_constraint)

[]

In [9]:
graph.query("SHOW CONSTRAINTS")

[{'id': 2,
  'name': 'etf_code_unique',
  'type': 'UNIQUENESS',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['code'],
  'ownedIndex': 'etf_code_unique',
  'propertyType': None}]

In [10]:
# 종목코드와 종목명은 반드시 존재해야 하는 필수 속성으로 설정 

exists_code_constraint = """
// 종목코드가 반드시 존재해야 하는 필수 속성으로 설정
CREATE CONSTRAINT etf_code_exists IF NOT EXISTS
FOR (e:ETF)
REQUIRE e.code IS NOT NULL
"""
graph.query(exists_code_constraint)

exists_name_constraint = """
// 종목명이 반드시 존재해야 하는 필수 속성으로 설정
CREATE CONSTRAINT etf_name_exists IF NOT EXISTS
FOR (e:ETF)
REQUIRE e.name IS NOT NULL
"""
graph.query(exists_name_constraint)

[]

In [11]:
graph.query("SHOW INDEXES")

[{'id': 5,
  'name': 'etf_code_unique',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['code'],
  'indexProvider': 'range-1.0',
  'owningConstraint': 'etf_code_unique',
  'lastRead': None,
  'readCount': None}]

`(3) Neo4J 그래프 DB에서 CSV파일 업로드`

In [12]:
df.head(1)

,종목코드,종목명,상장일,분류체계,운용사,수익률(최근 1년),기초지수,추적오차,순자산총액,괴리율,변동성,복제방법,총보수,과세유형
0,466400,1Q 25-08 회사채(A+이상)액티브,2023/09/19,채권-회사채-단기,하나자산운용,4.52,KIS 2025-08만기형 크레딧 A+이상 지수(총수익),0.11,111916276404,0.03,매우낮음,실물(액티브),0.1,배당소득세(보유기간과세)


In [13]:
# pandas의 to_dict를 활용한 데이터 변환
def clean_value(value):
    """NaN 값을 None으로 변환하는 헬퍼 함수"""
    if pd.isna(value):
        return None
    if isinstance(value, (int, float)) and pd.notna(value):
        return value
    return value

# DataFrame을 딕셔너리 리스트로 변환
etf_data = []
for record in df.to_dict('records'):
    cleaned_record = {k: clean_value(v) for k, v in record.items()}

    # 컬럼명을 영어로 매핑
    etf_record = {
        'code': cleaned_record['종목코드'],
        'name': cleaned_record['종목명'],
        'listingDate': cleaned_record['상장일'],
        'category': cleaned_record['분류체계'],
        'company': cleaned_record['운용사'],
        'yearReturn': cleaned_record['수익률(최근 1년)'],
        'baseIndex': cleaned_record['기초지수'],
        'trackingError': cleaned_record['추적오차'],
        'netAsset': cleaned_record['순자산총액'],
        'disparityRatio': cleaned_record['괴리율'],
        'volatility': cleaned_record['변동성'],
        'replicationMethod': cleaned_record['복제방법'],
        'totalFee': cleaned_record['총보수'],
        'taxType': cleaned_record['과세유형']
    }
    etf_data.append(etf_record)

# 배치 처리 실행
BATCH_SIZE = 100
batch_query = """
// ETF 노드 생성
UNWIND $etf_list AS etf    // $etf_list를 etf로 풀어헤치기
CREATE (:ETF {
    code: etf.code, // 종목코드
    name: etf.name, // 종목명
    listingDate: etf.listingDate, // 상장일
    category: etf.category, // 분류체계
    company: etf.company, // 운용사
    yearReturn: etf.yearReturn, // 수익률(최근 1년)
    baseIndex: etf.baseIndex,  // 기초지수
    trackingError: etf.trackingError, // 추적오차
    netAsset: etf.netAsset, // 순자산총액
    disparityRatio: etf.disparityRatio, // 괴리율
    volatility: etf.volatility, // 변동성
    replicationMethod: etf.replicationMethod, // 복제방법
    totalFee: etf.totalFee, // 총보수
    taxType: etf.taxType // 과세유형
})
"""

from tqdm import tqdm

# 배치 처리
for i in tqdm(range(0, len(etf_data), BATCH_SIZE)):
    batch = etf_data[i:i + BATCH_SIZE]
    graph.query(batch_query, params={'etf_list': batch})

print("ETF 데이터 배치 로드 완료")

100%|██████████| 10/10 [00:02<00:00,  4.94it/s]

ETF 데이터 배치 로드 완료


In [14]:
# ETF 노드 개수 확인
count_query = """
MATCH (e:ETF)   // 모든 ETF 노드 검색
RETURN count(e) AS count   // 노드 개수 반환
"""
graph.query(count_query)

[{'count': 930}]

In [15]:
# ETF 노드 속성으로 조건을 주고 검색
query = """
MATCH (e:ETF {code: '451060'})   // 종목코드가 451060인 ETF 노드 검색
RETURN e   // 노드 반환
"""
graph.query(query)

[{'e': {'trackingError': 0.77,
   'netAsset': 99754348820,
   'replicationMethod': '실물(액티브)',
   'listingDate': '2023/01/31',
   'code': '451060',
   'baseIndex': '코스피 200',
   'volatility': '높음',
   'yearReturn': -3.66,
   'disparityRatio': -0.01,
   'totalFee': 0.18,
   'name': '1Q K200액티브',
   'company': '하나자산운용',
   'category': '주식-시장대표',
   'taxType': '배당소득세(보유기간과세)'}}]

In [16]:
# 운용사(Company) 노드 생성 및 관계 설정
company_query = """
MATCH (e:ETF)       // ETF 노드에서
WITH DISTINCT e.company AS companyName    // 회사 이름을 가져옴
WHERE companyName IS NOT NULL             // 회사 이름이 NULL이 아닌 경우
MERGE (c:Company {name: companyName})     // Company 노드 생성
RETURN count(c) AS company_count          // 생성된 Company 노드 개수 반환
"""

result = graph.query(company_query)
print(result)

[{'company_count': 26}]


In [17]:
# ETF와 운용사 간의 관계 생성
relationship_query = """
MATCH (e:ETF), (c:Company)     // ETF와 Company 노드 모두 선택
WHERE e.company = c.name       // 회사 이름이 일치하는 경우
MERGE (c)-[r:MANAGES]->(e)      // Company 노드에서 ETF 노드로 MANAGES 관계 생성
RETURN count(r) AS relationship_count   // 생성된 관계 개수 반환
"""
graph.query(relationship_query)

[{'relationship_count': 930}]

`(4) 관계 추가`

In [18]:
# 분류체계(Category) 유일성 제약조건 설정
unique_category_constraint = """
CREATE CONSTRAINT category_name_unique IF NOT EXISTS
FOR (c:Category)
REQUIRE c.name IS UNIQUE
"""
graph.query(unique_category_constraint)

[]

In [19]:
# 분류체계(Category) 노드 생성
category_query = """
MATCH (e:ETF)           // ETF 노드에서
WITH DISTINCT e.category AS categoryName    // 분류체계 이름을 가져옴
WHERE categoryName IS NOT NULL              // 분류체계 이름이 NULL이 아닌 경우
MERGE (c:Category {name: categoryName})     // Category 노드 생성
RETURN count(c) as CategoryCount            // 생성된 Category 노드 수 반환
"""
graph.query(category_query)

[{'CategoryCount': 51}]

In [20]:
# ETF와 분류체계(Category) 간의 관계 생성
relationship_query = """
MATCH (e:ETF), (c:Category)     // ETF와 Category 노드 모두 선택
WHERE e.category = c.name       // 분류체계 이름이 일치하는 경우
MERGE (e)-[r:BELONGS_TO]->(c)    // ETF 노드에서 Category 노드로 BELONGS_TO 관계 생성
RETURN count(r) as RelationshipCount   // 생성된 관계 수 반환
"""
graph.query(relationship_query)

[{'RelationshipCount': 930}]

In [21]:
# 특정 카테코리의 ETF 노드 조회
category_query = """
MATCH (c:Category {name: '채권-혼합-단기'})<-[:BELONGS_TO]-(etf:ETF)  // '채권-혼합-단기' 카테고리에 속하는 ETF 노드 검색
RETURN etf.name, etf.yearReturn, c.name   // ETF 노드의 이름과 수익률 반환, 카테고리 반환
ORDER BY etf.yearReturn DESC // 수익률 기준 내림차순 정렬
LIMIT 5  // 상위 5개 노드만 반환
"""

graph.query(category_query)

[{'etf.name': '히어로즈 25-09 미국채권(AA-이상)액티브',
  'etf.yearReturn': 13.48,
  'c.name': '채권-혼합-단기'},
 {'etf.name': 'RISE KP달러채권액티브', 'etf.yearReturn': 13.06, 'c.name': '채권-혼합-단기'},
 {'etf.name': '히어로즈 단기채권ESG액티브', 'etf.yearReturn': 4.46, 'c.name': '채권-혼합-단기'},
 {'etf.name': 'SOL 초단기채권액티브', 'etf.yearReturn': 4.08, 'c.name': '채권-혼합-단기'},
 {'etf.name': 'RISE 머니마켓액티브', 'etf.yearReturn': 4.06, 'c.name': '채권-혼합-단기'}]

In [22]:
# 카테코리별 평균 수익률 등의 통계 데이터를 계산
category_stats_query = """
MATCH (c:Category)<-[:BELONGS_TO]-(etf:ETF)  // 모든 카테고리와 그에 속하는 ETF 노드 검색
RETURN c.name AS category,      // 카테고리 이름
       COUNT(etf) AS etf_count,    // ETF 개수
       AVG(etf.yearReturn) AS avg_yearReturn,   // 평균 수익률
       SUM(etf.netAsset) AS total_netAsset,     // 총 순자산총액
       AVG(etf.trackingError) AS avg_trackingError   // 평균 추적오차
ORDER BY avg_yearReturn DESC // 평균 수익률 기준 내림차순 정렬  
"""

category_stats = graph.query(category_stats_query)

# 결과를 DataFrame으로 변환
category_stats_df = pd.DataFrame(category_stats)
category_stats_df.head(10)  # 상위 10개 카테고리 출력

,category,etf_count,avg_yearReturn,total_netAsset,avg_trackingError
0,주식-업종섹터-중공업,1,54.010000,81792728257,0.380000
1,주식-업종섹터-커뮤니케이션서비스,2,31.140000,18572956872,0.580000
2,주식-업종섹터,6,29.843333,3200641966315,1.328333
3,주식-업종섹터-금융,8,27.668750,437772219204,1.782500
4,주식-업종섹터-헬스케어,11,24.427273,659858048321,2.348182
5,원자재-금속-금,5,23.880000,833097385377,0.638000
6,원자재-금속-은,1,20.880000,84014629668,1.180000
7,주식-업종섹터-산업재,5,17.974000,75641418942,1.186000
8,혼합자산-주식+채권,40,14.978250,2433229932286,1.801000
9,혼합자산,2,12.515000,148329575339,1.290000


## 3. GraphCypherQAChain 활용

In [23]:
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph

# LangChain 도구 활용 - LLM 및 그래프 객체 초기화
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0)

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE"),
    enhanced_schema=True,
)

# LangChain 도구 활용 - GraphCypherQAChain 객체 초기화
chain = GraphCypherQAChain.from_llm(
    llm=llm, 
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,)

result = chain.run("케이비자산운용은 모두 몇개의 ETF를 운용하고 있나요?")

C:\Users\kaydash\AppData\Local\Temp\ipykernel_14956\4211526367.py:22: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = chain.run("케이비자산운용은 모두 몇개의 ETF를 운용하고 있나요?")




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "케이비자산운용"})-[:MANAGES]->(e:ETF)
RETURN count(e) AS etfCount
Full Context:
[{'etfCount': 118}]

> Finished chain.


In [24]:
# 결과 출력
print(result)

케이비자산운용은 총 118개의 ETF를 운용하고 있습니다.


케이비, 운용, ETF

In [25]:
chain.run("케이비운용은 모두 몇개의 ETF를 운용하고 있나요?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "케이비운용"})-[:MANAGES]->(e:ETF)
RETURN count(e) AS etfCount
Full Context:
[{'etfCount': 0}]

> Finished chain.


'케이비운용은 총 0개의 ETF를 운용하고 있습니다.'

In [26]:
result = chain.invoke("코스피 200을 기초 지수로 하는 ETF는 무엇인가요?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (e:ETF) WHERE e.baseIndex CONTAINS "코스피 200" RETURN e.name
Full Context:
[{'e.name': 'KODEX 200'}, {'e.name': 'KOSEF 200'}, {'e.name': 'TIGER 200'}, {'e.name': 'ACE 200'}, {'e.name': 'TREX 200'}, {'e.name': 'KODEX 인버스'}, {'e.name': 'KODEX 레버리지'}, {'e.name': 'TIGER 인버스'}, {'e.name': 'TIGER 레버리지'}, {'e.name': 'TIGER 200 건설'}]

> Finished chain.


In [27]:
# 결과 출력
print(result)

{'query': '코스피 200을 기초 지수로 하는 ETF는 무엇인가요?', 'result': '코스피 200을 기초 지수로 하는 ETF는 KODEX 200, KOSEF 200, TIGER 200, ACE 200, TREX 200 등이 있습니다.'}


In [28]:
print(result['result'])

코스피 200을 기초 지수로 하는 ETF는 KODEX 200, KOSEF 200, TIGER 200, ACE 200, TREX 200 등이 있습니다.


## 4. 인덱스 생성
- 자주 쿼리하는 속성에 대해 인덱스를 생성하여 쿼리 성능 개선

`(1) 기본 인덱스`
- 유일성 제약조건이 있는 속성은 자동으로 인덱스가 생성되므로 별도 생성 불필요


In [29]:
# 인덱스 확인
graph.query("SHOW INDEXES")

[{'id': 8,
  'name': 'category_name_unique',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Category'],
  'properties': ['name'],
  'indexProvider': 'range-1.0',
  'owningConstraint': 'category_name_unique',
  'lastRead': neo4j.time.DateTime(2025, 12, 2, 23, 38, 34, 567000000, tzinfo=<UTC>),
  'readCount': 1084},
 {'id': 5,
  'name': 'etf_code_unique',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['code'],
  'indexProvider': 'range-1.0',
  'owningConstraint': 'etf_code_unique',
  'lastRead': neo4j.time.DateTime(2025, 12, 2, 23, 38, 34, 248000000, tzinfo=<UTC>),
  'readCount': 935}]

`(2) 인덱스 활용한 검색`

- 인덱스를 생성한 속성을 WHERE 절에서 사용하면 Neo4j가 자동으로 인덱스를 활용

In [30]:
# 종목코드로 ETF 검색 (정확한 일치 검색)
code_search_query = """
MATCH (e:ETF)
WHERE e.code = '451060'
RETURN e AS etf
"""
graph.query(code_search_query)

[{'etf': {'trackingError': 0.77,
   'netAsset': 99754348820,
   'replicationMethod': '실물(액티브)',
   'listingDate': '2023/01/31',
   'code': '451060',
   'baseIndex': '코스피 200',
   'volatility': '높음',
   'yearReturn': -3.66,
   'disparityRatio': -0.01,
   'totalFee': 0.18,
   'name': '1Q K200액티브',
   'company': '하나자산운용',
   'category': '주식-시장대표',
   'taxType': '배당소득세(보유기간과세)'}}]

In [31]:
# 종목코드로 ETF 검색 (범위 검색)
code_search_query = """
MATCH (e:ETF)
WHERE toFloat(e.code) >= 453000 AND toFloat(e.code) < 454000   // 코드가 문자열 순으로 '453'와 '454' 사이에 있는 ETF 검색
RETURN e.code AS etf_code, e.name AS etf_name  // 종목코드와 종목명 반환     
ORDER BY e.code ASC       // 종목코드 기준으로 정렬 (ASC/DESC)
LIMIT 10     // 최대 10개 결과 반환
"""
results = graph.query(code_search_query)

results = pd.DataFrame(results)
results

,etf_code,etf_name
0,453010,PLUS KOFR금리
1,453060,HANARO KOFR금리액티브(합성)
2,453080,KOSEF 미국나스닥100(H)
3,453330,RISE 미국S&P500(H)
4,453540,TIGER 25-10 회사채(A+이상)액티브
5,453630,KODEX 미국S&P500필수소비재
6,453640,KODEX 미국S&P500헬스케어
7,453650,KODEX 미국S&P500금융
8,453660,KODEX 미국S&P500경기소비재
9,453810,KODEX 인도Nifty50


In [32]:
print(graph.schema)

Node properties:
- **Company**
  - `name`: STRING Example: "미래에셋자산운용"
- **ETF**
  - `category`: STRING Example: "혼합자산-주식+채권"
  - `name`: STRING Example: "TIGER 엔비디아미국채커버드콜밸런스(합성)"
  - `code`: STRING Example: "0000D0"
  - `trackingError`: FLOAT Min: 0.0, Max: 44.42
  - `netAsset`: INTEGER Min: 89080687, Max: 9591343573385
  - `replicationMethod`: STRING Available options: ['합성(패시브)', '실물(패시브)', '실물(액티브)', '합성(액티브)']
  - `listingDate`: STRING Example: "2024/12/17"
  - `baseIndex`: STRING Example: "KEDI 엔비디아미국채30년타겟커버드콜혼합지수(TR)"
  - `volatility`: STRING Available options: ['매우낮음', '높음', '매우높음', '낮음', '보통']
  - `yearReturn`: FLOAT Min: -75.66, Max: 207.87
  - `disparityRatio`: FLOAT Min: -2.87, Max: 18010.07
  - `totalFee`: FLOAT Min: 0.0, Max: 0.99
  - `company`: STRING Example: "미래에셋자산운용"
  - `taxType`: STRING Available options: ['배당소득세(보유기간과세)', '비과세', '배당소득세(해외주식투자전용ETF)', '배당소득세(분리과세부동산ETF)', '비과세(분리과세부동산ETF)']
- **Category**
  - `name`: STRING Example: "주식-업종섹터-중공업"
Relationship prop

In [33]:
# 종목명으로 ETF 검색 (접두사 검색)
name_search_query = """
MATCH (e:ETF)  
WHERE e.name STARTS WITH 'TIGER'     // 'TIGER'로 시작하는 종목명 검색
RETURN e.name
"""

graph.query(name_search_query)

[{'e.name': 'TIGER 엔비디아미국채커버드콜밸런스(합성)'},
 {'e.name': 'TIGER 은행'},
 {'e.name': 'TIGER 반도체'},
 {'e.name': 'TIGER 방송통신'},
 {'e.name': 'TIGER 200'},
 {'e.name': 'TIGER 라틴35'},
 {'e.name': 'TIGER 국채3년'},
 {'e.name': 'TIGER 차이나항셍25'},
 {'e.name': 'TIGER 인버스'},
 {'e.name': 'TIGER 레버리지'},
 {'e.name': 'TIGER 원유선물Enhanced(H)'},
 {'e.name': 'TIGER 미국나스닥100'},
 {'e.name': 'TIGER 농산물선물Enhanced(H)'},
 {'e.name': 'TIGER 삼성그룹펀더멘털'},
 {'e.name': 'TIGER LG그룹+펀더멘털'},
 {'e.name': 'TIGER 현대차그룹+펀더멘털'},
 {'e.name': 'TIGER 200 건설'},
 {'e.name': 'TIGER 200 중공업'},
 {'e.name': 'TIGER 200 철강소재'},
 {'e.name': 'TIGER 200 에너지화학'},
 {'e.name': 'TIGER 200 IT'},
 {'e.name': 'TIGER 200 금융'},
 {'e.name': 'TIGER 경기방어'},
 {'e.name': 'TIGER 200 경기소비재'},
 {'e.name': 'TIGER 금속선물(H)'},
 {'e.name': 'TIGER 금은선물(H)'},
 {'e.name': 'TIGER 미국S&P500선물(H)'},
 {'e.name': 'TIGER 헬스케어'},
 {'e.name': 'TIGER 모멘텀'},
 {'e.name': 'TIGER 중국소비테마'},
 {'e.name': 'TIGER 단기통안채'},
 {'e.name': 'TIGER 소프트웨어'},
 {'e.name': 'TIGER 증권'},
 {'e.name': 'TIG

In [34]:
# 운용사 이름으로 ETF 검색 (IN 연산자 활용)
company_search_query = """
MATCH (e:ETF)  
WHERE e.company IN ['삼성자산운용', '미래에셋자산운용']  // 삼성자산운용 또는 미래에셋자산운용이 운용하는 ETF 검색" 
RETURN e.name, e.company  // 종목명과 운용사 이름 반환
LIMIT 10   // 최대 10개 결과 반환
"""

graph.query(company_search_query)

[{'e.name': 'TIGER 엔비디아미국채커버드콜밸런스(합성)', 'e.company': '미래에셋자산운용'},
 {'e.name': 'KODEX 200', 'e.company': '삼성자산운용'},
 {'e.name': 'KODEX 반도체', 'e.company': '삼성자산운용'},
 {'e.name': 'KODEX 은행', 'e.company': '삼성자산운용'},
 {'e.name': 'KODEX 자동차', 'e.company': '삼성자산운용'},
 {'e.name': 'TIGER 은행', 'e.company': '미래에셋자산운용'},
 {'e.name': 'TIGER 반도체', 'e.company': '미래에셋자산운용'},
 {'e.name': 'TIGER 방송통신', 'e.company': '미래에셋자산운용'},
 {'e.name': 'KODEX 차이나H', 'e.company': '삼성자산운용'},
 {'e.name': 'KODEX 일본TOPIX100', 'e.company': '삼성자산운용'}]

In [35]:
# 카테고리 노드와 관련된 관계 검색
relationship_query = """
MATCH (c:Category)      
WHERE c.name = '주식-시장대표'   // 카테고리 이름이 '주식-시장대표'인 카테고리 노드 검색
MATCH (e:ETF)-[r:BELONGS_TO]->(c)   // ETF 노드와 카테고리 노드 간의 BELONGS_TO 관계 검색
RETURN c.name AS category_name,  // 카테고리 이름 반환
       collect(e.name) AS etf_names,  // 관련된 ETF 노드의 이름을 리스트로 반환
       count(e) AS etf_count  // 관련된 ETF 개수 반환
"""

graph.query(relationship_query)

[{'category_name': '주식-시장대표',
  'etf_names': ['RISE 차이나HSCEI(H)',
   'RISE 차이나H선물인버스(H)',
   'RISE 코리아밸류업',
   'RISE 코스닥150',
   'RISE 코스닥150선물레버리지',
   'RISE 코스닥150선물인버스',
   'RISE 코스피',
   'SOL 200TR',
   'SOL KRX300',
   'SOL 미국S&P500',
   'SOL 미국S&P500ESG',
   'TIGER 200',
   'SOL 차이나강소기업CSI500(합성 H)',
   'SOL 차이나육성산업액티브(합성)',
   'SOL 코스닥150',
   'TIGER 200선물인버스2X',
   'TIGER AI코리아그로스액티브',
   'TIGER KRX300',
   'TIGER 글로벌이노베이션액티브',
   'TIGER 라틴35',
   'TIGER 레버리지',
   'TIGER 미국S&P500레버리지(합성 H)',
   'TIGER 미국S&P500선물인버스(H)',
   'TIGER 미국나스닥100ETF선물',
   'TIGER 미국나스닥100레버리지(합성)',
   'TIGER 유로스탁스50(합성 H)',
   'TIGER 인도니프티50',
   'TIGER 인도니프티50레버리지(합성)',
   'TIGER 일본TOPIX(합성 H)',
   'TIGER 일본니케이225',
   'TIGER 차이나CSI300레버리지(합성)',
   'TIGER 차이나HSCEI',
   'TIGER 차이나항셍25',
   'TIGER 코스닥150',
   'TIGER 코스닥150선물인버스',
   'TIGER 코스피',
   'TIMEFOLIO 코리아밸류업액티브',
   'TREX 200',
   'TRUSTON 주주가치액티브',
   'TRUSTON 코리아밸류업액티브',
   'UNICORN R&D 액티브',
   'WON 200',
   '마이티 코스피100',
   '마이티 다이나믹퀀트액티브',


`(3) 인덱스 생성`

- 대량의 데이터를 처리하거나 자주 쿼리하는 속성에 대해 인덱스를 생성하면 쿼리 성능이 크게 향상됨

In [36]:
# 수익률에 대한 인덱스 생성 - 수익률 기반 검색 및 정렬 시 성능 향상
return_index_query = """
CREATE INDEX etf_yearreturn_idx IF NOT EXISTS
FOR (e:ETF) ON (e.yearReturn)
"""
graph.query(return_index_query)

[]

In [37]:
# 수익률 기준으로 ETF 검색 (정렬)
return_search_query = """
MATCH (e:ETF)
WHERE e.yearReturn > 0.1   // 수익률이 10% 이상인 ETF 검색
RETURN e.name AS Name, e.yearReturn AS Return   // 종목명과 수익률 반환
ORDER BY e.yearReturn DESC   // 수익률 기준으로 내림차순 정렬
LIMIT 10   // 최대 10개 결과 반환
"""

graph.query(return_search_query)

[{'Name': 'ACE 미국빅테크TOP7 Plus레버리지(합성)', 'Return': 207.87},
 {'Name': 'PLUS 미국테크TOP10레버리지(합성)', 'Return': 179.94},
 {'Name': 'TIGER 미국나스닥100레버리지(합성)', 'Return': 106.09},
 {'Name': 'TIMEFOLIO 미국나스닥100액티브', 'Return': 97.75},
 {'Name': 'TIMEFOLIO 글로벌AI인공지능액티브', 'Return': 91.95},
 {'Name': 'HANARO 글로벌생성형AI액티브', 'Return': 88.45},
 {'Name': 'ACE 미국빅테크TOP7 Plus', 'Return': 85.06},
 {'Name': '에셋플러스 글로벌플랫폼액티브', 'Return': 84.22},
 {'Name': 'KODEX 미국메타버스나스닥액티브', 'Return': 80.51},
 {'Name': 'TIGER 글로벌AI액티브', 'Return': 77.62}]

In [38]:
# 순자산총액에 대한 인덱스 생성 - 규모별 검색 및 정렬 시 성능 향상
netasset_index_query = """
CREATE INDEX etf_netasset_idx IF NOT EXISTS
FOR (e:ETF) ON (e.netAsset)
"""
graph.query(netasset_index_query)

[]

In [39]:
# 순자산총액 기준으로 ETF 검색 (정렬)
netasset_search_query = """
MATCH (e:ETF)
WHERE e.netAsset > 1000000000   // 순자산총액이 10억 이상인 ETF 검색
RETURN e.name, e.netAsset   // 종목명과 순자산총액 반환
ORDER BY e.netAsset DESC   // 순자산총액 기준으로 내림차순 정렬
LIMIT 10   // 최대 10개 결과 반환
"""
graph.query(netasset_search_query)

[{'e.name': 'KODEX CD금리액티브(합성)', 'e.netAsset': 9591343573385},
 {'e.name': 'TIGER CD금리투자KIS(합성)', 'e.netAsset': 6871517745155},
 {'e.name': 'TIGER 미국S&P500', 'e.netAsset': 6433623514411},
 {'e.name': 'KODEX 200', 'e.netAsset': 5604796801979},
 {'e.name': 'TIGER 미국나스닥100', 'e.netAsset': 4494225408877},
 {'e.name': 'KODEX KOFR금리액티브(합성)', 'e.netAsset': 4331602537465},
 {'e.name': 'KODEX 머니마켓액티브', 'e.netAsset': 4004995260449},
 {'e.name': 'TIGER KOFR금리액티브(합성)', 'e.netAsset': 3477409947171},
 {'e.name': 'TIGER 미국테크TOP10 INDXX', 'e.netAsset': 3144375360249},
 {'e.name': 'KODEX 미국S&P500TR', 'e.netAsset': 3050770738959}]

`(4) 복합 인덱스`

- 여러 속성을 함께 검색하는 경우가 많다면 복합 인덱스를 고려

In [40]:
# 복합 인덱스 생성 - 카테고리와 수익률을 동시에 검색 및 정렬 시 성능 향상
compound_index_query = """
CREATE INDEX etf_category_return_idx IF NOT EXISTS
FOR (e:ETF) ON (e.category, e.yearReturn)
"""
graph.query(compound_index_query)

[]

In [41]:
# 카테고리와 수익률 기준으로 ETF 검색 (정렬)
compound_search_query = """
MATCH (e:ETF)
WHERE e.category = '주식-시장대표' AND e.yearReturn > 0.1   // 카테고리가 '주식-시장대표'이고 수익률이 10% 이상인 ETF 검색
RETURN e.name, e.yearReturn   // 종목명과 수익률 반환
ORDER BY e.yearReturn DESC   // 수익률 기준으로 내림차순 정렬
LIMIT 10   // 최대 10개 결과 반환
"""
graph.query(compound_search_query)

[{'e.name': 'TIGER 미국나스닥100레버리지(합성)', 'e.yearReturn': 106.09},
 {'e.name': 'TIMEFOLIO 미국나스닥100액티브', 'e.yearReturn': 97.75},
 {'e.name': '에셋플러스 글로벌플랫폼액티브', 'e.yearReturn': 84.22},
 {'e.name': 'TIMEFOLIO 미국S&P500액티브', 'e.yearReturn': 71.73},
 {'e.name': 'KODEX 미국나스닥100레버리지(합성 H)', 'e.yearReturn': 63.12},
 {'e.name': '에셋플러스 글로벌대장장이액티브', 'e.yearReturn': 59.98},
 {'e.name': 'TIGER 미국S&P500레버리지(합성 H)', 'e.yearReturn': 54.0},
 {'e.name': 'TIGER 차이나CSI300레버리지(합성)', 'e.yearReturn': 50.87},
 {'e.name': 'KODEX 미국나스닥100TR', 'e.yearReturn': 48.77},
 {'e.name': 'ACE 중국본토CSI300레버리지(합성)', 'e.yearReturn': 48.62}]

`(5) Full Text 검색 인덱스`

- 텍스트 검색이 필요한 경우  Full Text 인덱스를 고려

In [42]:
# Full Text 없이 종목명 검색 ("액티브" 포함)
name_search_query = """
MATCH (e:ETF)  
WHERE e.name CONTAINS '액티브'   // 종목명에 '액티브'가 포함된 ETF 검색
RETURN e.name   // 종목명 반환
LIMIT 10   // 최대 10개 결과 반환
"""
graph.query(name_search_query)

[{'e.name': 'RISE 단기국공채액티브'},
 {'e.name': 'RISE 중장기국공채액티브'},
 {'e.name': 'TIGER 단기채권액티브'},
 {'e.name': 'ACE 중장기국공채액티브'},
 {'e.name': 'KODEX 종합채권(AA-이상)액티브'},
 {'e.name': 'KODEX 단기변동금리부채권액티브'},
 {'e.name': 'PLUS 단기채권액티브'},
 {'e.name': 'TIGER 미국달러단기채권액티브'},
 {'e.name': 'RISE 금융채액티브'},
 {'e.name': 'ACE 종합채권(AA-이상)KIS액티브'}]

In [43]:
# Full Text 인덱스 생성 - 종목명에 대한 Full Text 검색을 위한 인덱스 생성
fulltext_index_query = """
CREATE FULLTEXT INDEX etf_name_fulltext IF NOT EXISTS
FOR (e:ETF) ON EACH [e.name]
"""
graph.query(fulltext_index_query)

[]

In [44]:
# Full Text 검색 - 종목명에 "액티브" 포함
fulltext_search_query = """
CALL db.index.fulltext.queryNodes('etf_name_fulltext', '액티브') YIELD node  // Full Text 인덱스 검색
RETURN node.name   // 종목명 반환
LIMIT 10   // 최대 10개 결과 반환
"""
graph.query(fulltext_search_query)

[{'node.name': 'UNICORN R&D 액티브'},
 {'node.name': 'KODEX 1년은행양도성예금증서+액티브(합성)'},
 {'node.name': 'TIGER 종합채권(AA-이상)액티브'},
 {'node.name': 'KODEX ESG종합채권(A-이상)액티브'},
 {'node.name': '히어로즈 종합채권(AA-이상)액티브'},
 {'node.name': 'KODEX 장기종합채권(AA-이상)액티브'},
 {'node.name': 'WON 종합채권(AA-이상)액티브'},
 {'node.name': 'KODEX 종합채권(AA-이상)액티브'},
 {'node.name': 'PLUS 종합채권(AA-이상)액티브'},
 {'node.name': 'SOL 종합채권(AA-이상)액티브'}]

In [45]:
graph.query("SHOW FULLTEXT INDEXES")  # 생성된 Full Text 인덱스 확인

[{'id': 9,
  'name': 'etf_name_fulltext',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['name'],
  'indexProvider': 'fulltext-2.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': None}]

In [46]:
# # 인덱스 삭제
# drop_index_query = """
# DROP INDEX etf_name_baseindex_fulltext IF EXISTS
# """
# graph.query(drop_index_query)

In [47]:
# Full Text 인덱스 생성 - 종목명, 기초지수에 대한 Full Text 검색을 위한 인덱스 생성 (cjk 형태소 분석기 사용)
fulltext_index_query = """
CREATE FULLTEXT INDEX etf_name_baseindex_fulltext IF NOT EXISTS
FOR (e:ETF) ON EACH [e.name, e.baseIndex]
OPTIONS {
    indexConfig: {
        `fulltext.analyzer`: 'cjk',   // cjk 형태소 분석기 사용 
        `fulltext.eventually_consistent`: true  // 성능 최적화를 위한 설정 활성화
    }
}
"""
graph.query(fulltext_index_query)

[]

In [48]:
graph.query("SHOW FULLTEXT INDEXES")  # 생성된 Full Text 인덱스 확인

[{'id': 1,
  'name': 'etf_name_baseindex_fulltext',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['name', 'baseIndex'],
  'indexProvider': 'fulltext-2.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': None},
 {'id': 9,
  'name': 'etf_name_fulltext',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['name'],
  'indexProvider': 'fulltext-2.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': None}]

In [49]:
# Full Text 검색 - 종목명 또는 기초지수에 "클린 에너지" 포함
fulltext_search_query = """
CALL db.index.fulltext.queryNodes('etf_name_baseindex_fulltext', '클린 에너지') YIELD node  // Full Text 인덱스 검색
RETURN node.name, node.baseIndex   // 종목명과 기초지수 반환
LIMIT 10   // 최대 10개 결과 반환
"""
graph.query(fulltext_search_query)

[{'node.name': 'KODEX 에너지화학', 'node.baseIndex': 'KRX 에너지화학'},
 {'node.name': 'TIGER 200 에너지화학', 'node.baseIndex': '코스피 200 에너지/화학'},
 {'node.name': 'KODEX K-신재생에너지액티브',
  'node.baseIndex': 'FnGuide K-신재생에너지 플러스 지수'},
 {'node.name': 'RISE 글로벌클린에너지',
  'node.baseIndex': 'S&P Global Clean Energy Index (Price Return)'},
 {'node.name': 'KODEX 미국클린에너지나스닥',
  'node.baseIndex': 'Nasdaq Clean Edge Green Energy Index (Price Return)'},
 {'node.name': 'TIGER 200에너지화학레버리지', 'node.baseIndex': '코스피 200 에너지/화학'},
 {'node.name': 'TIGER Fn신재생에너지', 'node.baseIndex': 'FnGuide 신재생에너지 지수'},
 {'node.name': 'HANARO Fn친환경에너지',
  'node.baseIndex': 'FnGuide 친환경에너지 지수 (시장가격)'},
 {'node.name': 'KOSEF 미국원유에너지기업',
  'node.baseIndex': 'MSCI US IMI Energy 25/50 Index(Price Return)'}]

`(6) 벡터 인덱스 검색`

- 벡터 검색이 필요한 경우 벡터 인덱스를 고려
- 단일 속성에 대해서만 벡터 인덱스를 지원

In [50]:
from langchain_core.documents import Document

etfs = graph.query("""
MATCH (e:ETF)   // 모든 ETF 노드 검색
RETURN e.name AS name,   // 종목명 반환
       e.company AS company,   // 운용사 반환
       e.baseIndex AS baseIndex,   // 기초지수 반환
       e.category AS category,   // 카테고리 반환
       e.listingDate AS listingDate   // 상장일 반환
""")

# 검색 결과를 DataFrame으로 변환
etfs_df = pd.DataFrame(etfs)

# 종목명, 운용사, 기초지수 속성을 결합한 문서를 생성
docs = [
    Document(
        page_content=f"종목명: {row['name']}, 운용사: {row['company']}, 기초지수: {row['baseIndex']}",
        metadata={"name": row['name'], "company": row['company'], "baseIndex": row['baseIndex'], "category": row['category'], "listingDate": row['listingDate']}
    )
    for _, row in etfs_df.iterrows()
]

# 문서의 속성 확인
for doc in docs[:5]:  # 상위 5개 문서 출력
    print(doc.page_content)
    print(doc.metadata)
    print()

종목명: TIGER 엔비디아미국채커버드콜밸런스(합성), 운용사: 미래에셋자산운용, 기초지수: KEDI 엔비디아미국채30년타겟커버드콜혼합지수(TR)
{'name': 'TIGER 엔비디아미국채커버드콜밸런스(합성)', 'company': '미래에셋자산운용', 'baseIndex': 'KEDI 엔비디아미국채30년타겟커버드콜혼합지수(TR)', 'category': '혼합자산-주식+채권', 'listingDate': '2024/12/17'}

종목명: KODEX 200, 운용사: 삼성자산운용, 기초지수: 코스피 200
{'name': 'KODEX 200', 'company': '삼성자산운용', 'baseIndex': '코스피 200', 'category': '주식-시장대표', 'listingDate': '2002/10/14'}

종목명: KOSEF 200, 운용사: 키움투자자산운용, 기초지수: 코스피 200
{'name': 'KOSEF 200', 'company': '키움투자자산운용', 'baseIndex': '코스피 200', 'category': '주식-시장대표', 'listingDate': '2002/10/14'}

종목명: KODEX 반도체, 운용사: 삼성자산운용, 기초지수: KRX 반도체
{'name': 'KODEX 반도체', 'company': '삼성자산운용', 'baseIndex': 'KRX 반도체', 'category': '주식-업종섹터-정보기술', 'listingDate': '2006/06/27'}

종목명: KODEX 은행, 운용사: 삼성자산운용, 기초지수: KRX 은행
{'name': 'KODEX 은행', 'company': '삼성자산운용', 'baseIndex': 'KRX 은행', 'category': '주식-업종섹터-금융', 'listingDate': '2006/06/27'}



In [51]:
from langchain_openai import OpenAIEmbeddings

# OpenAI 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 기존 ETF 데이터 조회
etfs = graph.query("""
MATCH (e:ETF)
RETURN e.name AS name,
       e.company AS company,
       e.baseIndex AS baseIndex,
       e.category AS category,
       e.listingDate AS listingDate
""")

# 각 ETF에 대해 임베딩 생성 및 업데이트
for etf in etfs:
    # ETF 정보를 텍스트로 결합
    combined_text = f"이 종목은 {etf['name']}이며, 운용사는 {etf['company']}입니다. 기초지수는 {etf['baseIndex']}입니다."
    
    # 임베딩 생성
    embedding_vector = embeddings.embed_query(combined_text)
    
    # 기존 ETF 노드에 임베딩 속성 추가
    graph.query("""
    MATCH (e:ETF {name: $name})
    CALL db.create.setNodeVectorProperty(e, 'embedding', $embedding)
    """, params={
        'name': etf['name'],
        'embedding': embedding_vector
    })

In [52]:
# 벡터 인덱스 생성
graph.query("""
CREATE VECTOR INDEX etf_vector_index IF NOT EXISTS
FOR (e:ETF) ON (e.embedding)
OPTIONS {
  indexConfig: {
    `vector.dimensions`: 1536,
    `vector.similarity_function`: 'cosine'
  }
}
""")

[]

In [53]:
# 기존 벡터 인덱스에 연결
from langchain_openai import OpenAIEmbeddings
from langchain_neo4j import Neo4jVector

embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 

vector_db = Neo4jVector.from_existing_index(
    embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name="etf_vector_index",  # 위에서 생성한 인덱스 이름
    node_label="ETF",  # 노드 레이블
    text_node_property="name",  # 텍스트 속성 
    embedding_node_property="embedding"  # 임베딩 속성
)

# 유사도 검색 테스트
results = vector_db.similarity_search("KODEX", k=5)
for result in results:
    print(f"ETF: {result.page_content}")
    print(f"메타데이터: {result.metadata}")
    print("---")

ETF: KODEX IT
메타데이터: {'trackingError': 0.52, 'netAsset': 18249222663, 'replicationMethod': '실물(패시브)', 'listingDate': '2017/03/28', 'code': '266370', 'baseIndex': 'KRX 정보기술', 'volatility': '매우높음', 'yearReturn': -17.87, 'disparityRatio': 0.05, 'totalFee': 0.45, 'company': '삼성자산운용', 'category': '주식-업종섹터-정보기술', 'taxType': '비과세'}
---
ETF: KODEX 미디어&엔터테인먼트
메타데이터: {'trackingError': 0.2, 'netAsset': 41406746008, 'replicationMethod': '실물(패시브)', 'listingDate': '2017/03/28', 'code': '266360', 'baseIndex': 'KRX 미디어&엔터테인먼트', 'volatility': '높음', 'yearReturn': -7.1, 'disparityRatio': 0.04, 'totalFee': 0.45, 'company': '삼성자산운용', 'category': '주식-업종섹터-정보기술', 'taxType': '비과세'}
---
ETF: KODEX 운송
메타데이터: {'trackingError': 0.81, 'netAsset': 13516501568, 'replicationMethod': '실물(패시브)', 'listingDate': '2011/04/26', 'code': '140710', 'baseIndex': 'KRX 운송', 'volatility': '높음', 'yearReturn': 7.31, 'disparityRatio': -0.32, 'totalFee': 0.45, 'company': '삼성자산운용', 'category': '주식-업종섹터-산업재', 'taxType': '비과세'}
---
ETF:

In [54]:
# from langchain_openai import OpenAIEmbeddings
# from langchain_neo4j import Neo4jVector

# embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 

# vector_db = Neo4jVector.from_documents(
#     docs,
#     embeddings,
#     url=NEO4J_URI,
#     username=NEO4J_USERNAME,
#     password=NEO4J_PASSWORD,
#     index_name="etf_name_company_asset_vector_index" # 인덱스 이름 설정 (선택 사항)
# )

In [55]:
query = "삼성자산운용의 국내 주식형 상품은 무엇인가요?" # 검색할 쿼리
similar_docs = vector_db.similarity_search_with_score(query, k=5) # 유사도 상위 5개 문서 검색

print(f"'{query}'와 유사한 문서:")
for doc, score in similar_docs:
    print(f"문서 내용: {doc.page_content}, 유사도 점수: {score}")
    print(f"메타데이터: {doc.metadata}")
    print("-" * 50)  # 구분선 출력

'삼성자산운용의 국내 주식형 상품은 무엇인가요?'와 유사한 문서:
문서 내용: KODEX 단기채권, 유사도 점수: 0.7644956707954407
메타데이터: {'trackingError': 0.03, 'netAsset': 687245148920, 'replicationMethod': '실물(패시브)', 'listingDate': '2012/02/22', 'code': '153130', 'baseIndex': 'KRW Cash 지수(총수익)', 'volatility': '매우낮음', 'yearReturn': 3.46, 'disparityRatio': 0.0, 'totalFee': 0.15, 'company': '삼성자산운용', 'category': '채권-국공채-단기', 'taxType': '배당소득세(보유기간과세)'}
--------------------------------------------------
문서 내용: KODEX 증권, 유사도 점수: 0.7558038234710693
메타데이터: {'trackingError': 1.47, 'netAsset': 31554064691, 'replicationMethod': '실물(패시브)', 'listingDate': '2008/05/29', 'code': '102970', 'baseIndex': 'KRX 증권', 'volatility': '매우높음', 'yearReturn': 15.29, 'disparityRatio': -0.31, 'totalFee': 0.45, 'company': '삼성자산운용', 'category': '주식-업종섹터-금융', 'taxType': '비과세'}
--------------------------------------------------
문서 내용: KODEX 삼성그룹, 유사도 점수: 0.7541424632072449
메타데이터: {'trackingError': 0.48, 'netAsset': 1019156170735, 'replicationMethod': '실물(패시브)', 

In [56]:
# 유사한 문서들의 메타데이터에서 카테고리와 운용사 정보 추출
categories = set()
companies = set()

for doc, score in similar_docs:
    # 벡터 검색된 문서의 메타데이터에서 정보 추출
    if 'category' in doc.metadata:
        categories.add(doc.metadata['category'])
    if 'company' in doc.metadata:
        companies.add(doc.metadata['company'])

# 쿼리에서 언급된 조건들도 직접 추가
target_company = "삼성자산운용"
target_categories = ["통화-미국달러", "채권-혼합-단기"]  # 가능한 카테고리 표현들

# ETF 노드에서 조건에 맞는 상품 검색 (특정 운용사가 관리하는 ETF 중 카테고리가 일치하는 상품)
etf_query = """
MATCH (e:ETF)
WHERE e.company = $target_company  // 자산운용사 매칭
  AND e.category IN $target_categories  // 카테고리 매칭
WITH e
ORDER BY toFloat(e.netAsset) DESC
LIMIT 10
RETURN 
    e.name AS name,
    e.company AS company,
    e.category AS category,
    e.baseIndex AS baseIndex,
    e.yearReturn AS yearReturn,
    e.totalFee AS totalFee,
    toFloat(e.netAsset) AS netAsset
"""

params = {
    "target_company": target_company,
    "target_categories": target_categories,
}

etf_results = graph.query(etf_query, params=params)
etf_df = pd.DataFrame(etf_results)
print(f"검색된 ETF 상품: {len(etf_df)}개")
etf_df.head(10)

검색된 ETF 상품: 7개


,name,company,category,baseIndex,yearReturn,totalFee,netAsset
0,KODEX 머니마켓액티브,삼성자산운용,채권-혼합-단기,KAP MMF 지수(TR),0.00,0.05,4.004995e+12
1,KODEX 단기채권PLUS,삼성자산운용,채권-혼합-단기,KRW Cash PLUS 지수(총수익),3.71,0.15,1.641191e+12
2,KODEX 단기변동금리부채권액티브,삼성자산운용,채권-혼합-단기,KAP 단기변동금리부은행채권지수,3.50,0.15,1.788297e+11
3,KODEX 미국달러선물인버스2X,삼성자산운용,통화-미국달러,미국달러선물지수,-21.48,0.41,1.551231e+11
4,KODEX 미국달러선물,삼성자산운용,통화-미국달러,미국달러선물지수,16.66,0.21,5.999443e+10
5,KODEX 미국달러선물인버스,삼성자산운용,통화-미국달러,미국달러선물지수,-9.85,0.41,4.743744e+10
6,KODEX 미국달러선물레버리지,삼성자산운용,통화-미국달러,미국달러선물지수,30.25,0.45,3.894008e+10


In [57]:
# ETF 전용 벡터 검색 설정
from langchain_openai import OpenAIEmbeddings
from langchain_neo4j import Neo4jVector

embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 

vector_db = Neo4jVector.from_existing_index(
    embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name="etf_vector_index",  # ETF 벡터 인덱스
    node_label="ETF",  # ETF 노드 레이블
    text_node_property="name",  # 기본 텍스트 속성 (ETF 이름)
    embedding_node_property="embedding",  # 임베딩 속성
    retrieval_query="""
    // 기본 ETF 노드와 점수
    WITH node, score
    
    // ETF와 관련된 모든 정보 수집
    OPTIONAL MATCH (node)-[:BELONGS_TO]->(category:Category)
    OPTIONAL MATCH (node)-[:MANAGED_BY]->(company:Company)
    
    // 관련 ETF들도 찾기 (같은 카테고리에 속하는)
    OPTIONAL MATCH (node)-[:BELONGS_TO]->(cat:Category)<-[:BELONGS_TO]-(related_etf:ETF)
    WHERE related_etf <> node       // 같은 ETF 제외
    
    // 집계 함수를 사용하여 관련 ETF들 수집
    WITH node, score,
         collect(DISTINCT related_etf.name)[0..3] AS related_etfs   // 관련 ETF 3개만 (중복 제외)
    
    // 검색용 종합 텍스트 생성
    WITH node, score, related_etfs,
         "ETF 종목: " + node.name + "\n" + 
         "- 운용사: " + COALESCE(node.company, "N/A") + "\n" +
         "- 기초지수: " + COALESCE(node.baseIndex, "N/A") + "\n" +
         "- 카테고리: " + COALESCE(node.category, "N/A") + "\n" +
         "- 관련 ETF:" + apoc.text.join(related_etfs, ", ") + "\n" AS combined_text

    RETURN combined_text AS text,   // 검색 결과 텍스트 (Langchain Document의 page_content 속성)
           score,   // 벡터 유사도 점수
           {
               etf_name: node.name,
               company: node.company,
               base_index: node.baseIndex,
               category: node.category,
               listing_date: node.listingDate,
               expense_ratio: node.expenseRatio,
               nav: node.nav,
               related_etfs: related_etfs,
               etf_code: node.code
           } AS metadata    // 메타데이터 (Langchain Document의 metadata 속성)
    """
)


# 유사도 검색 테스트
def search_etfs(query, k=5):
    """ETF 검색 함수"""
    results = vector_db.similarity_search(query, k=k)
    
    print(f"검색어: '{query}'")
    print(f"검색 결과 ({len(results)}개):")
    print("-" * 50)
    
    for i, result in enumerate(results, 1):
        metadata = result.metadata
        print(f"{i}.")
        print(f"{result.page_content}") # 페이지 내용 출력           
        print(f"상장일: {metadata.get('listing_date', 'N/A')}")
        print()


# 일반적인 카테고리 검색
search_etfs("국내 주식 ETF")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: MANAGED_BY)} {position: line: 7, column: 29, offset: 307} for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \n    // 기본 ETF 노드와 점수\n    WITH node, score\n\n    // ETF와 관련된 모든 정보 수집\n    OPTIONAL MATCH (node)-[:BELONGS_TO]->(category:Category)\n    OPTIONAL MATCH (node)-[:MANAGED_BY]->(company:Company)\n\n    // 관련 ETF들도 찾기 (같은 카테고리에 속하는)\n    OPTIONAL MATCH (node)-[:BELONGS_TO]->(cat:Category)<-[:BELONGS_TO]-(related_etf:ETF)

검색어: '국내 주식 ETF'
검색 결과 (5개):
--------------------------------------------------
1.
ETF 종목: KOSEF 미국원유에너지기업
- 운용사: 키움투자자산운용
- 기초지수: MSCI US IMI Energy 25/50 Index(Price Return)
- 카테고리: 주식-업종섹터-에너지화학
- 관련 ETF:TIGER 200에너지화학레버리지, KODEX 미국S&P500에너지(합성), KODEX 에너지화학

상장일: 2024/01/16

2.
ETF 종목: KODEX MSCI KOREA ESG유니버설
- 운용사: 삼성자산운용
- 기초지수: MSCI Korea ESG Universal Capped Index
- 카테고리: 주식-전략-전략테마
- 관련 ETF:RISE 플랫폼테마, SOL KRX기후변화솔루션, SOL 차이나태양광CSI(합성)

상장일: 2018/02/07

3.
ETF 종목: KOSEF 글로벌전력GRID인프라
- 운용사: 키움투자자산운용
- 기초지수: NASDAQ OMX Clean Edge Smart Grid Infrastructure Index
- 카테고리: 주식-업종섹터-업종테마
- 관련 ETF:RISE 차이나항셍테크, RISE 창업투자회사, RISE 컨택트대표

상장일: 2024/08/27

4.
ETF 종목: KODEX MSCI EM선물(H)
- 운용사: 삼성자산운용
- 기초지수: iEdge Emerging Markets Futures Index(ER)
- 카테고리: 주식-시장대표
- 관련 ETF:RISE 차이나HSCEI(H), RISE 차이나H선물인버스(H), RISE 코리아밸류업

상장일: 2018/03/23

5.
ETF 종목: KODEX 미국클린에너지나스닥
- 운용사: 삼성자산운용
- 기초지수: Nasdaq Clean Edge Green Energy Index (Price Return)
- 카테고리: 주식-업종섹터-업종테마
- 관련 ETF:RISE 차이나항셍테크, RISE 창업

In [58]:
# 특정 운용사 검색  
search_etfs("삼성자산운용 ETF")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: MANAGED_BY)} {position: line: 7, column: 29, offset: 307} for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \n    // 기본 ETF 노드와 점수\n    WITH node, score\n\n    // ETF와 관련된 모든 정보 수집\n    OPTIONAL MATCH (node)-[:BELONGS_TO]->(category:Category)\n    OPTIONAL MATCH (node)-[:MANAGED_BY]->(company:Company)\n\n    // 관련 ETF들도 찾기 (같은 카테고리에 속하는)\n    OPTIONAL MATCH (node)-[:BELONGS_TO]->(cat:Category)<-[:BELONGS_TO]-(related_etf:ETF)

검색어: '삼성자산운용 ETF'
검색 결과 (5개):
--------------------------------------------------
1.
ETF 종목: KODEX MSCI KOREA ESG유니버설
- 운용사: 삼성자산운용
- 기초지수: MSCI Korea ESG Universal Capped Index
- 카테고리: 주식-전략-전략테마
- 관련 ETF:RISE 플랫폼테마, SOL KRX기후변화솔루션, SOL 차이나태양광CSI(합성)

상장일: 2018/02/07

2.
ETF 종목: KODEX MSCI EM선물(H)
- 운용사: 삼성자산운용
- 기초지수: iEdge Emerging Markets Futures Index(ER)
- 카테고리: 주식-시장대표
- 관련 ETF:RISE 차이나HSCEI(H), RISE 차이나H선물인버스(H), RISE 코리아밸류업

상장일: 2018/03/23

3.
ETF 종목: TIGER MSCI KOREA ESG유니버설
- 운용사: 미래에셋자산운용
- 기초지수: MSCI Korea ESG Universal Index
- 카테고리: 주식-전략-전략테마
- 관련 ETF:RISE 플랫폼테마, SOL KRX기후변화솔루션, SOL 차이나태양광CSI(합성)

상장일: 2018/02/07

4.
ETF 종목: KODEX MSCI밸류
- 운용사: 삼성자산운용
- 기초지수: MSCI Korea IMI Enhanced Value Capped
- 카테고리: 주식-전략-가치
- 관련 ETF:TIGER 우량가치, VITA 밸류알파액티브, ACE 미국WideMoat동일가중

상장일: 2017/07/11

5.
ETF 종목: KODEX MSCI Korea
- 운용사: 삼성자산운용
- 기초지수: MSCI Korea Index
- 카테고리: 주식-시장대표
- 관련 ETF:RISE 차이나HSCEI(H), RISE 차이나H선물인버스(H), RISE 코리아밸류업

상장일: 2012/04/30



In [59]:
# 특정 지수 추적 ETF 검색
search_etfs("코스피200 추적 ETF")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: MANAGED_BY)} {position: line: 7, column: 29, offset: 307} for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \n    // 기본 ETF 노드와 점수\n    WITH node, score\n\n    // ETF와 관련된 모든 정보 수집\n    OPTIONAL MATCH (node)-[:BELONGS_TO]->(category:Category)\n    OPTIONAL MATCH (node)-[:MANAGED_BY]->(company:Company)\n\n    // 관련 ETF들도 찾기 (같은 카테고리에 속하는)\n    OPTIONAL MATCH (node)-[:BELONGS_TO]->(cat:Category)<-[:BELONGS_TO]-(related_etf:ETF)

검색어: '코스피200 추적 ETF'
검색 결과 (5개):
--------------------------------------------------
1.
ETF 종목: KOSEF 200
- 운용사: 키움투자자산운용
- 기초지수: 코스피 200
- 카테고리: 주식-시장대표
- 관련 ETF:RISE 차이나HSCEI(H), RISE 차이나H선물인버스(H), RISE 코리아밸류업

상장일: 2002/10/14

2.
ETF 종목: KOSEF 200선물인버스
- 운용사: 키움투자자산운용
- 기초지수: 코스피 200 선물지수
- 카테고리: 주식-시장대표
- 관련 ETF:RISE 차이나HSCEI(H), RISE 차이나H선물인버스(H), RISE 코리아밸류업

상장일: 2016/09/12

3.
ETF 종목: KOSEF 200선물인버스2X
- 운용사: 키움투자자산운용
- 기초지수: 코스피 200 선물지수
- 카테고리: 주식-시장대표
- 관련 ETF:RISE 차이나HSCEI(H), RISE 차이나H선물인버스(H), RISE 코리아밸류업

상장일: 2016/09/22

4.
ETF 종목: KOSEF 200TR
- 운용사: 키움투자자산운용
- 기초지수: 코스피 200 TR
- 카테고리: 주식-시장대표
- 관련 ETF:RISE 차이나HSCEI(H), RISE 차이나H선물인버스(H), RISE 코리아밸류업

상장일: 2018/04/23

5.
ETF 종목: KOSEF 200선물레버리지
- 운용사: 키움투자자산운용
- 기초지수: 코스피 200 선물지수
- 카테고리: 주식-시장대표
- 관련 ETF:RISE 차이나HSCEI(H), RISE 차이나H선물인버스(H), RISE 코리아밸류업

상장일: 2016/09/12



In [60]:
# 배당 관련 ETF 검색
search_etfs("배당 ETF")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: MANAGED_BY)} {position: line: 7, column: 29, offset: 307} for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \n    // 기본 ETF 노드와 점수\n    WITH node, score\n\n    // ETF와 관련된 모든 정보 수집\n    OPTIONAL MATCH (node)-[:BELONGS_TO]->(category:Category)\n    OPTIONAL MATCH (node)-[:MANAGED_BY]->(company:Company)\n\n    // 관련 ETF들도 찾기 (같은 카테고리에 속하는)\n    OPTIONAL MATCH (node)-[:BELONGS_TO]->(cat:Category)<-[:BELONGS_TO]-(related_etf:ETF)

검색어: '배당 ETF'
검색 결과 (5개):
--------------------------------------------------
1.
ETF 종목: KOSEF 미국방어배당성장나스닥
- 운용사: 키움투자자산운용
- 기초지수: Nasdaq US Low Volatility Dividend Achievers Index
- 카테고리: 주식-전략-배당
- 관련 ETF:RISE 중소형고배당, SOL 금융지주플러스고배당, SOL 미국배당다우존스

상장일: 2020/12/24

2.
ETF 종목: TIGER 미국배당다우존스
- 운용사: 미래에셋자산운용
- 기초지수: Dow Jones U.S. Dividend 100 Price Return Index
- 카테고리: 주식-전략-배당
- 관련 ETF:RISE 중소형고배당, SOL 금융지주플러스고배당, SOL 미국배당다우존스

상장일: 2023/06/20

3.
ETF 종목: TIGER 유로스탁스배당30
- 운용사: 미래에셋자산운용
- 기초지수: Euro STOXX Select Dividend 30
- 카테고리: 주식-전략-배당
- 관련 ETF:RISE 중소형고배당, SOL 금융지주플러스고배당, SOL 미국배당다우존스

상장일: 2016/07/01

4.
ETF 종목: KOSEF 고배당
- 운용사: 키움투자자산운용
- 기초지수: MKF 웰스 고배당20
- 카테고리: 주식-전략-배당
- 관련 ETF:RISE 중소형고배당, SOL 금융지주플러스고배당, SOL 미국배당다우존스

상장일: 2008/07/29

5.
ETF 종목: ACE 미국배당다우존스
- 운용사: 한국투자신탁운용
- 기초지수: Dow Jones U.S. Dividend 100 Price Return Index
- 카테고리: 주식-전략-배당
- 관련 ETF:RISE 중소형고배당, SOL 금융지주플러스고배당, SOL 미국배당다우존스

상장일: 2021/10/21



## 5. Text to Cypher
- LangChain으로 Neo4J 지식 그래프 조회
- Ollama 모델 사용

`(1) GraphCypherQAChain 설정`

In [61]:
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph

# LangChain 도구 활용 - LLM 및 그래프 객체 초기화
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0)

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE"),
    enhanced_schema=True,
    refresh_schema=True,  # 스키마를 최신 상태로 유지
)

# LangChain 도구 활용 - GraphCypherQAChain 객체 초기화
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm, 
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,)

`(2) Text to Cypher - DB 조회`

In [62]:
cypher_chain.invoke({"query": "삼성자산운용의 KODEX 관련 상품은 무엇인가요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF) WHERE e.name CONTAINS "KODEX" RETURN e.name, e.code, e.category, e.trackingError, e.netAsset, e.replicationMethod, e.listingDate, e.baseIndex, e.volatility, e.yearReturn, e.disparityRatio, e.totalFee, e.taxType
Full Context:
[{'e.name': 'KODEX 200', 'e.code': '069500', 'e.category': '주식-시장대표', 'e.trackingError': 0.69, 'e.netAsset': 5604796801979, 'e.replicationMethod': '실물(패시브)', 'e.listingDate': '2002/10/14', 'e.baseIndex': '코스피 200', 'e.volatility': '높음', 'e.yearReturn': -5.08, 'e.disparityRatio': -0.04, 'e.totalFee': 0.15, 'e.taxType': '비과세'}, {'e.name': 'KODEX 반도체', 'e.code': '091160', 'e.category': '주식-업종섹터-정보기술', 'e.trackingError': 0.49, 'e.netAsset': 474340847022, 'e.replicationMethod': '실물(패시브)', 'e.listingDate': '2006/06/27', 'e.baseIndex': 'KRX 반도체', 'e.volatility': '매우높음', 'e.yearReturn': -15.81, 'e.disparityRatio': -0.27, 'e.totalFee': 0.45, 'e.taxType': '비과

{'query': '삼성자산운용의 KODEX 관련 상품은 무엇인가요?',
 'result': '삼성자산운용의 KODEX 관련 상품으로는 KODEX 200, KODEX 반도체, KODEX 은행, KODEX 자동차, KODEX 차이나H, KODEX 일본TOPIX100, KODEX 삼성그룹, KODEX 기계장비, KODEX 증권, KODEX 국고채3년 등이 있습니다.'}

`(3) 출력 갯수를 지정 (top k)`

In [63]:
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    top_k=3,
)

cypher_chain.invoke({"query": "삼성자산운용의 KODEX 관련 상품은 무엇인가요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF) WHERE e.name CONTAINS "KODEX" RETURN e.name
Full Context:
[{'e.name': 'KODEX 200'}, {'e.name': 'KODEX 반도체'}, {'e.name': 'KODEX 은행'}]

> Finished chain.


{'query': '삼성자산운용의 KODEX 관련 상품은 무엇인가요?',
 'result': '삼성자산운용의 KODEX 관련 상품은 KODEX 200, KODEX 반도체, KODEX 은행입니다.'}

`(4) 중간 결과를 포함하여 출력`

In [64]:
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    top_k=3,
    return_intermediate_steps=True
)

cypher_chain.invoke({"query": "삼성자산운용의 KODEX 관련 상품은 무엇인가요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF) WHERE e.name CONTAINS "KODEX" RETURN e.name
Full Context:
[{'e.name': 'KODEX 200'}, {'e.name': 'KODEX 반도체'}, {'e.name': 'KODEX 은행'}]

> Finished chain.


{'query': '삼성자산운용의 KODEX 관련 상품은 무엇인가요?',
 'result': '삼성자산운용의 KODEX 관련 상품은 KODEX 200, KODEX 반도체, KODEX 은행입니다.',
 'intermediate_steps': [{'query': 'MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF) WHERE e.name CONTAINS "KODEX" RETURN e.name'},
  {'context': [{'e.name': 'KODEX 200'},
    {'e.name': 'KODEX 반도체'},
    {'e.name': 'KODEX 은행'}]}]}

`(5) cypher 쿼리 결과를 직접 출력 (LLM 답변 생성하지 않음)`

In [65]:
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    top_k=3,
    return_intermediate_steps=True,
    return_direct=True
)

cypher_chain.invoke({"query": "삼성자산운용의 KODEX 관련 상품은 무엇인가요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF) WHERE e.name CONTAINS "KODEX" RETURN e.name, e.code

> Finished chain.


{'query': '삼성자산운용의 KODEX 관련 상품은 무엇인가요?',
 'result': [{'e.name': 'KODEX 200', 'e.code': '069500'},
  {'e.name': 'KODEX 반도체', 'e.code': '091160'},
  {'e.name': 'KODEX 은행', 'e.code': '091170'}],
 'intermediate_steps': [{'query': 'MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF) WHERE e.name CONTAINS "KODEX" RETURN e.name, e.code'}]}

`(6) cypher 쿼리 생성하는 모델과 최종 답변 생성 모델을 별도 적용`

In [66]:
# 원격 Ollama 서버 연결 (test_ollama.py 방식)
from langchain_openai import ChatOpenAI

# Ollama 서버 설정
OLLAMA_BASE_URL = "http://littletask.kro.kr:1410/v1"
OLLAMA_API_KEY = "Task123!"
OLLAMA_MODEL = "phi4-mini:3.8b"

# OpenAI 호환 API를 사용하여 원격 Ollama 서버에 연결
second_llm = ChatOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=OLLAMA_API_KEY,
    model=OLLAMA_MODEL,
    temperature=0.0
)

print("✅ second_llm 정의 완료")

# 연결 테스트 (선택사항)
try:
    test_response = second_llm.invoke("안녕하세요")
    print(f"✅ Ollama 서버 연결 성공: {test_response.content[:50]}")
except Exception as e:
    print(f"⚠️ Ollama 연결 테스트 실패: {str(e)[:100]}")
    print("💡 second_llm은 정의되었지만, 실제 사용 시 오류가 발생할 수 있습니다.")


✅ second_llm 정의 완료
✅ Ollama 서버 연결 성공: 안녕하세요! 어떻게 도와드릴까요?


`(7) 사용자 프롬프트를 직접 지정`

In [67]:
from langchain_core.prompts.prompt import PromptTemplate

# Cypher 생성을 위한 프롬프트
CYPHER_GENERATION_TEMPLATE = """Task: Generate Cypher statement to query a graph database.
Instructions:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided.
Schema:
{schema}

Note: Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
Do not include any text except the generated Cypher statement.

Examples: Here are a few examples of generated Cypher statements for particular questions:
# 수익률이 가장 높은 5개 ETF는 무엇인가요?
MATCH (e:ETF)
WHERE e.yearReturn IS NOT NULL
RETURN e.name, e.code, e.yearReturn
ORDER BY e.yearReturn DESC
LIMIT 5

# 미래에셋자산운용에서 운용하는 모든 ETF를 보여주세요.
MATCH (c:Company {{name: '미래에셋자산운용'}})-[:MANAGES]->(e:ETF)
RETURN e.name, e.code, e.category, e.yearReturn
ORDER BY e.yearReturn DESC

# 국내 주식형 ETF 중 순자산총액이 가장 큰 상위 10개는 무엇인가요?
MATCH (c:Category {{name: '국내 주식형'}})<-[:BELONGS_TO]-(e:ETF)
WHERE e.netAsset IS NOT NULL
RETURN e.name, e.code, e.netAsset, e.yearReturn
ORDER BY e.netAsset DESC
LIMIT 10

# 순자산총액이 1조원 이상인 ETF의 평균 수익률은 얼마인가요?
MATCH (e:ETF)
WHERE e.netAsset >= 1000000000000 AND e.yearReturn IS NOT NULL
RETURN AVG(e.yearReturn) AS averageReturn, COUNT(e) AS etfCount

# 운용사별 평균 ETF 수익률은 어떻게 되나요?
MATCH (c:Company)-[:MANAGES]->(e:ETF)
WHERE e.yearReturn IS NOT NULL
WITH c.name AS company, AVG(e.yearReturn) AS avgReturn, COUNT(e) AS etfCount
ORDER BY avgReturn DESC
RETURN company, avgReturn, etfCount

The question is:
{question}"""

# 결과 처리를 위한 QA 프롬프트
QA_TEMPLATE = """
You are a financial assistant providing information about ETF databases in an easy-to-understand manner in Korean.
Based on the information obtained from the ETF database, answer the question in a clear and informative way.

Question: {question}
Search Results: {context}

Respond only with relevant information in a natural, conversational tone. Convert numerical data such as returns and net asset values into appropriate units (%, million USD, billion USD, etc.) to make them easily readable.
Avoid content that could be interpreted as investment advice or recommendations, and only convey objective facts.
"""

CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"], 
    template=CYPHER_GENERATION_TEMPLATE
)

QA_PROMPT = PromptTemplate(
    input_variables=["question", "context"], 
    template=QA_TEMPLATE
)

# Chain 생성 - input_key와 output_key를 명시적으로 설정
cypher_chain = GraphCypherQAChain.from_llm(
    cypher_llm=llm,
    qa_llm=second_llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    cypher_prompt=CYPHER_GENERATION_PROMPT,
    qa_prompt=QA_PROMPT,
    input_key="question",  
    output_key="result"
)

# 쿼리 실행 - 입력 키를 "question"으로 변경
cypher_chain.invoke({"question": "삼성자산운용의 KODEX 관련 상품은 무엇인가요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: '삼성자산운용'})-[:MANAGES]->(e:ETF)
WHERE e.name CONTAINS 'KODEX'
RETURN e.name, e.code, e.category, e.yearReturn
ORDER BY e.yearReturn DESC
Full Context:
[{'e.name': 'KODEX 미국메타버스나스닥액티브', 'e.code': '411420', 'e.category': '주식-업종섹터-업종테마', 'e.yearReturn': 80.51}, {'e.name': 'KODEX 미국나스닥100레버리지(합성 H)', 'e.code': '409820', 'e.category': '주식-시장대표', 'e.yearReturn': 63.12}, {'e.name': 'KODEX 미국반도체MV', 'e.code': '390390', 'e.category': '주식-업종섹터-업종테마', 'e.yearReturn': 61.51}, {'e.name': 'KODEX 미국S&P500커뮤니케이션', 'e.code': '463690', 'e.category': '주식-업종섹터-커뮤니케이션서비스', 'e.yearReturn': 58.86}, {'e.name': 'KODEX 미국빅테크10(H)', 'e.code': '314250', 'e.category': '주식-업종섹터-업종테마', 'e.yearReturn': 57.08}, {'e.name': 'KODEX 테슬라밸류체인FactSet', 'e.code': '459560', 'e.category': '주식-업종섹터-업종테마', 'e.yearReturn': 54.65}, {'e.name': 'KODEX 미국나스닥100TR', 'e.code': '379810', 'e.category': '주식-시장대표', 'e.yearReturn': 48.77}, {'e.name': 'KODEX

{'question': '삼성자산운용의 KODEX 관련 상품은 무엇인가요?',
 'result': "삼성자산운용의 KODEX 관련 상품 중 하나는 'KODEX 미국메타버스나스닥액티브'입니다. 이 상품의 코드는 411420이며, 주식-업종섹터-업종테마에 속합니다. 이 상품의 연간 리턴은 80.51%로, 이는 매년 평균으로 8% 이상을 의미합니다.\n\n이 정보는 투자자들이 삼성자산운용의 KODEX 관련 상품에 대한 이해를 돕기 위해 제공된 것입니다. 그러나 이 데이터는 투자 결정을 내리는 데 도움이 되지 않으며, 투자 결정을 내리려면 전문가의 조언과 종합적인 분석이 필요합니다.\n\n참고: 리턴은 연간 평균으로 계산되며, 실제로는 매일 변동할 수 있습니다. 또한, 'e.yearReturn' 데이터가 80.51%로 표시된 경우, 이는 8% 이상을 의미하며, 이는 일반적으로 좋은 리턴이라고 간주됩니다. 그러나 이 정보만으로는 투자 결정을 내리는 데 충분하지 않습니다.\n\n이 정보를 참고하여 더 자세한 정보를 원하시면, ETF 데이터베이스나 삼성자산운용의 공식 웹사이트를 참조하는 것을 추천합니다."}

## 6. 벡터 검색(Semantic Search) 

`(1) Graph DB를 초기화`

In [77]:
from langchain_openai import OpenAIEmbeddings
from langchain_neo4j import Neo4jVector

embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 

existing_graph = Neo4jVector.from_existing_index(
    embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name="etf_vector_index",  # 기존에 생성한 벡터 인덱스 이름
    node_label="ETF",  # 노드 레이블
    text_node_property="name",  # 텍스트 속성 (ETF 이름)
    embedding_node_property="embedding",  # 임베딩 속성
    retrieval_query="""
    WITH node, score
    WITH node, score,
         "ETF 종목: " + node.name + 
         CASE WHEN node.company IS NOT NULL THEN "\n- 운용사: " + node.company ELSE "" END +
         CASE WHEN node.baseIndex IS NOT NULL THEN "\n- 기초지수: " + node.baseIndex ELSE "" END +
         CASE WHEN node.category IS NOT NULL THEN "\n- 카테고리: " + node.category ELSE "" END AS combined_text
         
    RETURN combined_text AS text,   // 텍스트 (LangChain Document 객체의 page_content 속성)
           score,  // 유사도 점수 
           {
               etf_name: node.name,
               company: node.company,
               base_index: node.baseIndex,
               category: node.category,
               listing_date: node.listingDate,
               total_fee: node.totalFee,
               etf_code: node.code
           } AS metadata    // 메타데이터 (LangChain Document 객체의 metadata 속성)
    """
)

# 벡터 검색 쿼리 실행
query = "삼성자산운용의 KODEX 관련 상품은 무엇인가요?" # 검색할 쿼리
similar_docs = existing_graph.similarity_search_with_score(query, k=5) # 유사도 상위 5개 문서 검색

for doc, score in similar_docs:
    print(f"문서 내용: {doc.page_content}, 유사도 점수: {score}")

`(2) Graph DB 검색을 수행하는 함수`

In [78]:
# Cypher 쿼리 실행 및 관련 ETF 노드 검색을 처리하는 함수 
def execute_query_and_get_etf_data(query, k=5):
    # 벡터 검색 쿼리 실행
    similar_docs = existing_graph.similarity_search_with_score(query, k) # 유사도 상위 k개 문서 검색
    similar_doc_names = [doc.metadata['etf_name'] for doc, _ in similar_docs] # 유사한 문서의 종목명 추출

    # 유사한 문서의 종목명을 기반으로 ETF 노드 검색
    etf_query = """
    MATCH (e:ETF)   // 모든 ETF 노드 검색
    WHERE e.name IN $similar_doc_names   // 유사한 문서의 종목명과 일치하는 ETF 노드 검색
    RETURN e
    """
    etf_results = graph.query(
        etf_query, 
        params={"similar_doc_names": similar_doc_names}
    ) # ETF 노드 검색
    etf_results = [etf['e'] for etf in etf_results] # 검색된 ETF 노드 반환

    # 문자열 포맷팅
    result = ""
    for etf in etf_results:
        result += f"# 종목명: {etf['name']} (운용사: {etf['company']})\n"
        result += f"- 기초지수: {etf['baseIndex']}, 수익률: {etf['yearReturn']}, 순자산총액: {etf['netAsset']}\n"
        result += f"- 추적오차: {etf['trackingError']}, 변동성: {etf['volatility']}, 복제방법: {etf['replicationMethod']}, 총보수: {etf['totalFee']}, 과세유형: {etf['taxType']}\n"
        result += "-" * 3 + "\n"
    return result


# 쿼리 실행 및 관련 ETF 노드 검색
query = "삼성자산운용의 KODEX 관련 상품은 무엇인가요?" 
etf_data = execute_query_and_get_etf_data(query, k=5) # ETF 노드 검색
print(etf_data)

`(3) LCEL 사용하여 RAG 체인을 구성`

In [79]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# LLM
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0)

# Prompt
template = '''당신은 ETF 데이터 분석 전문가로서 오직 주어진 정보에 기반하여 객관적이고 정확한 답변을 제공합니다.

주어진 정보:
{context}

질문: {question}

답변 작성 지침:
1. 제공된 정보에 명시된 사실만 사용하세요.
2. 수익률은 '%', 순자산총액은 '억 원' 또는 '조 원' 단위로 표시하세요.
3. 투자 조언이나 추천으로 해석될 수 있는 표현은 사용하지 마세요.
4. 제공된 정보에 없는 내용은 "제공된 정보에서 해당 내용을 찾을 수 없습니다"라고 답하세요.
5. 한국어로 자연스럽고 이해하기 쉽게 답변하세요.
'''

prompt = ChatPromptTemplate.from_template(template)

# RAG Chain 연결
rag_chain = (
    {'context': RunnableLambda(execute_query_and_get_etf_data), 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Chain 실행
query = "삼성자산운용의 KODEX 관련 상품은 무엇인가요?" 
answer = rag_chain.invoke(query)

print(answer)

제공된 정보에서 해당 내용을 찾을 수 없습니다.


### **[실습]** 

- S&P 500 회사 데이터(data/sp500_companies.csv)를 가져와서, Knowledge Graph로 구현합니다. 
- 관계 구조: Company → Sector → Industry (또는 Company → Exchange)
- 지식 그래프 검색의 다양한 기법을 적용합니다. 

- 출처: https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks?resource=download

#### 1. 데이터 로드 및 전처리

In [80]:
sp500_df = pd.read_csv("./data/sp500_companies.csv")
sp500_df.head()

,Exchange,Symbol,Shortname,Longname,Sector,Industry,Currentprice,Marketcap,Ebitda,Revenuegrowth,City,State,Country,Fulltimeemployees,Longbusinesssummary,Weight
0,NMS,AAPL,Apple Inc.,Apple Inc.,Technology,Consumer Electronics,254.49,3846819807232,1.346610e+11,0.061,Cupertino,CA,United States,164000.0,"Apple Inc. designs, manufactures, and markets ...",0.069209
1,NMS,NVDA,NVIDIA Corporation,NVIDIA Corporation,Technology,Semiconductors,134.70,3298803056640,6.118400e+10,1.224,Santa Clara,CA,United States,29600.0,NVIDIA Corporation provides graphics and compu...,0.059350
2,NMS,MSFT,Microsoft Corporation,Microsoft Corporation,Technology,Software - Infrastructure,436.60,3246068596736,1.365520e+11,0.160,Redmond,WA,United States,228000.0,Microsoft Corporation develops and supports so...,0.058401
3,NMS,AMZN,"Amazon.com, Inc.","Amazon.com, Inc.",Consumer Cyclical,Internet Retail,224.92,2365033807872,1.115830e+11,0.110,Seattle,WA,United States,1551000.0,"Amazon.com, Inc. engages in the retail sale of...",0.042550
4,NMS,GOOGL,Alphabet Inc.,Alphabet Inc.,Communication Services,Internet Content & Information,191.41,2351625142272,1.234700e+11,0.151,Mountain View,CA,United States,181269.0,Alphabet Inc. offers various products and plat...,0.042309


#### 2. 제약 조건 설정

In [81]:
# ==========================================
# 데이터베이스 초기화 (S&P 500 실습 시작 전)
# ==========================================
print("⚠️ 데이터베이스 초기화 중...")
graph.query("MATCH (n) DETACH DELETE n")
print("✅ 데이터베이스 초기화 완료")

⚠️ 데이터베이스 초기화 중...
✅ 데이터베이스 초기화 완료


In [82]:
# 여기에 코드를 작성하세요.

# Company 노드의 Symbol이 고유하도록 유일성 제약조건 설정
unique_symbol_constraint = """
CREATE CONSTRAINT company_symbol_unique IF NOT EXISTS
FOR (c:Company)
REQUIRE c.Symbol IS UNIQUE
"""
graph.query(unique_symbol_constraint)

# Company 노드의 Symbol이 반드시 존재하도록 설정
exists_symbol_constraint = """
CREATE CONSTRAINT company_symbol_exists IF NOT EXISTS
FOR (c:Company)
REQUIRE c.Symbol IS NOT NULL
"""
graph.query(exists_symbol_constraint)

# Sector 노드의 name이 고유하도록 유일성 제약조건 설정
unique_sector_constraint = """
CREATE CONSTRAINT sector_name_unique IF NOT EXISTS
FOR (s:Sector)
REQUIRE s.name IS UNIQUE
"""
graph.query(unique_sector_constraint)

# Industry 노드의 name이 고유하도록 유일성 제약조건 설정
unique_industry_constraint = """
CREATE CONSTRAINT industry_name_unique IF NOT EXISTS
FOR (i:Industry)
REQUIRE i.name IS UNIQUE
"""
graph.query(unique_industry_constraint)

# Exchange 노드의 name이 고유하도록 유일성 제약조건 설정
unique_exchange_constraint = """
CREATE CONSTRAINT exchange_name_unique IF NOT EXISTS
FOR (e:Exchange)
REQUIRE e.name IS UNIQUE
"""
graph.query(unique_exchange_constraint)

print("✅ 제약 조건 설정 완료")
graph.query("SHOW CONSTRAINTS")

✅ 제약 조건 설정 완료


[{'id': 4,
  'name': 'category_name_unique',
  'type': 'UNIQUENESS',
  'entityType': 'NODE',
  'labelsOrTypes': ['Category'],
  'properties': ['name'],
  'ownedIndex': 'category_name_unique',
  'propertyType': None},
 {'id': 14,
  'name': 'company_symbol_exists',
  'type': 'NODE_PROPERTY_EXISTENCE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Company'],
  'properties': ['Symbol'],
  'ownedIndex': None,
  'propertyType': None},
 {'id': 12,
  'name': 'company_symbol_unique',
  'type': 'UNIQUENESS',
  'entityType': 'NODE',
  'labelsOrTypes': ['Company'],
  'properties': ['Symbol'],
  'ownedIndex': 'company_symbol_unique',
  'propertyType': None},
 {'id': 13,
  'name': 'etf_code_exists',
  'type': 'NODE_PROPERTY_EXISTENCE',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['code'],
  'ownedIndex': None,
  'propertyType': None},
 {'id': 2,
  'name': 'etf_code_unique',
  'type': 'UNIQUENESS',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['code'],
  

### 3. 노드 및 관계 생성

In [83]:
# 여기에 코드를 작성하세요.

# DataFrame을 딕셔너리 리스트로 변환
def clean_value(value):
    """NaN 값을 None으로 변환하는 헬퍼 함수"""
    if pd.isna(value):
        return None
    return value

# 회사 데이터 변환
company_data = []
for record in sp500_df.to_dict('records'):
    cleaned_record = {k: clean_value(v) for k, v in record.items()}
    company_data.append(cleaned_record)

print(f"총 {len(company_data)}개의 회사 데이터 준비 완료")

# 1. Company 노드 생성
company_batch_query = """
UNWIND $company_list AS company
CREATE (:Company {
    Symbol: company.Symbol,
    Security: company.Security,
    Sector: company.`GICS Sector`,
    SubIndustry: company.`GICS Sub-Industry`,
    HeadquartersLocation: company.`Headquarters Location`,
    DateAdded: company.`Date added`,
    CIK: company.CIK,
    Founded: company.Founded
})
"""

BATCH_SIZE = 100
from tqdm import tqdm

for i in tqdm(range(0, len(company_data), BATCH_SIZE), desc="Company 노드 생성"):
    batch = company_data[i:i + BATCH_SIZE]
    graph.query(company_batch_query, params={'company_list': batch})

print("✅ Company 노드 생성 완료")

# 2. Sector 노드 생성
sector_query = """
MATCH (c:Company)
WITH DISTINCT c.Sector AS sectorName
WHERE sectorName IS NOT NULL
MERGE (s:Sector {name: sectorName})
RETURN count(s) AS sector_count
"""
result = graph.query(sector_query)
print(f"✅ Sector 노드 생성 완료: {result[0]['sector_count']}개")

# 3. Industry 노드 생성
industry_query = """
MATCH (c:Company)
WITH DISTINCT c.SubIndustry AS industryName
WHERE industryName IS NOT NULL
MERGE (i:Industry {name: industryName})
RETURN count(i) AS industry_count
"""
result = graph.query(industry_query)
print(f"✅ Industry 노드 생성 완료: {result[0]['industry_count']}개")

# 4. Exchange 노드 생성 (Symbol에서 추출)
exchange_query = """
MATCH (c:Company)
WITH DISTINCT 
    CASE 
        WHEN c.Symbol CONTAINS '.' THEN split(c.Symbol, '.')[1]
        ELSE 'NYSE'
    END AS exchangeName
MERGE (e:Exchange {name: exchangeName})
RETURN count(e) AS exchange_count
"""
result = graph.query(exchange_query)
print(f"✅ Exchange 노드 생성 완료: {result[0]['exchange_count']}개")

# 5. Company와 Sector 관계 생성
company_sector_rel = """
MATCH (c:Company), (s:Sector)
WHERE c.Sector = s.name
MERGE (c)-[r:BELONGS_TO_SECTOR]->(s)
RETURN count(r) AS rel_count
"""
result = graph.query(company_sector_rel)
print(f"✅ Company-Sector 관계 생성 완료: {result[0]['rel_count']}개")

# 6. Company와 Industry 관계 생성
company_industry_rel = """
MATCH (c:Company), (i:Industry)
WHERE c.SubIndustry = i.name
MERGE (c)-[r:IN_INDUSTRY]->(i)
RETURN count(r) AS rel_count
"""
result = graph.query(company_industry_rel)
print(f"✅ Company-Industry 관계 생성 완료: {result[0]['rel_count']}개")

# 7. Industry와 Sector 관계 생성
industry_sector_rel = """
MATCH (c:Company)-[:IN_INDUSTRY]->(i:Industry)
MATCH (c)-[:BELONGS_TO_SECTOR]->(s:Sector)
WITH DISTINCT i, s
MERGE (i)-[r:PART_OF_SECTOR]->(s)
RETURN count(r) AS rel_count
"""
result = graph.query(industry_sector_rel)
print(f"✅ Industry-Sector 관계 생성 완료: {result[0]['rel_count']}개")

# 8. Company와 Exchange 관계 생성
company_exchange_rel = """
MATCH (c:Company), (e:Exchange)
WHERE (c.Symbol CONTAINS '.' AND split(c.Symbol, '.')[1] = e.name)
   OR (NOT c.Symbol CONTAINS '.' AND e.name = 'NYSE')
MERGE (c)-[r:LISTED_ON]->(e)
RETURN count(r) AS rel_count
"""
result = graph.query(company_exchange_rel)
print(f"✅ Company-Exchange 관계 생성 완료: {result[0]['rel_count']}개")

print("\n✅ 모든 노드 및 관계 생성 완료")

총 502개의 회사 데이터 준비 완료


Company 노드 생성: 100%|██████████| 6/6 [00:01<00:00,  4.74it/s]


✅ Company 노드 생성 완료
✅ Sector 노드 생성 완료: 0개
✅ Industry 노드 생성 완료: 0개
✅ Exchange 노드 생성 완료: 1개
✅ Company-Sector 관계 생성 완료: 0개
✅ Company-Industry 관계 생성 완료: 0개
✅ Industry-Sector 관계 생성 완료: 0개
✅ Company-Exchange 관계 생성 완료: 502개

✅ 모든 노드 및 관계 생성 완료


### 4. 데이터 확인 및 검색

In [84]:
# 여기에 코드를 작성하세요.

# 1. 전체 노드 개수 확인
node_count_query = """
MATCH (n)
RETURN labels(n)[0] AS nodeType, count(n) AS count
ORDER BY count DESC
"""
result = graph.query(node_count_query)
print("=== 노드 타입별 개수 ===")
for record in result:
    print(f"{record['nodeType']}: {record['count']}개")

print("\n" + "="*50 + "\n")

# 2. 특정 회사 검색 (예: Apple)
company_search_query = """
MATCH (c:Company {Symbol: 'AAPL'})
RETURN c.Symbol AS Symbol, 
       c.Security AS Name,
       c.Sector AS Sector,
       c.SubIndustry AS Industry,
       c.HeadquartersLocation AS Location,
       c.Founded AS Founded
"""
result = graph.query(company_search_query)
print("=== Apple Inc. 정보 ===")
if result:
    record = result[0]
    for key, value in record.items():
        print(f"{key}: {value}")

print("\n" + "="*50 + "\n")

# 3. 섹터별 회사 수 통계
sector_stats_query = """
MATCH (s:Sector)<-[:BELONGS_TO_SECTOR]-(c:Company)
RETURN s.name AS Sector, count(c) AS CompanyCount
ORDER BY CompanyCount DESC
"""
result = graph.query(sector_stats_query)
print("=== 섹터별 회사 수 ===")
for record in result:
    print(f"{record['Sector']}: {record['CompanyCount']}개")

print("\n" + "="*50 + "\n")

# 4. 특정 섹터의 회사 목록 조회 (예: Information Technology)
sector_companies_query = """
MATCH (c:Company)-[:BELONGS_TO_SECTOR]->(s:Sector {name: 'Information Technology'})
RETURN c.Symbol AS Symbol, c.Security AS Name
ORDER BY c.Symbol
LIMIT 10
"""
result = graph.query(sector_companies_query)
print("=== Information Technology 섹터 회사 (상위 10개) ===")
for record in result:
    print(f"{record['Symbol']}: {record['Name']}")

print("\n" + "="*50 + "\n")

# 5. 관계 패턴 검색: Company -> Industry -> Sector 경로
relationship_pattern_query = """
MATCH (c:Company)-[:IN_INDUSTRY]->(i:Industry)-[:PART_OF_SECTOR]->(s:Sector)
WHERE c.Symbol = 'AAPL'
RETURN c.Security AS Company,
       i.name AS Industry,
       s.name AS Sector
"""
result = graph.query(relationship_pattern_query)
print("=== Apple의 관계 구조 ===")
if result:
    record = result[0]
    print(f"회사: {record['Company']}")
    print(f"산업: {record['Industry']}")
    print(f"섹터: {record['Sector']}")

print("\n" + "="*50 + "\n")

# 6. 거래소별 회사 수
exchange_stats_query = """
MATCH (e:Exchange)<-[:LISTED_ON]-(c:Company)
RETURN e.name AS Exchange, count(c) AS CompanyCount
ORDER BY CompanyCount DESC
"""
result = graph.query(exchange_stats_query)
print("=== 거래소별 회사 수 ===")
for record in result:
    print(f"{record['Exchange']}: {record['CompanyCount']}개")

print("\n" + "="*50 + "\n")

# 7. 특정 산업의 회사 목록
industry_companies_query = """
MATCH (c:Company)-[:IN_INDUSTRY]->(i:Industry {name: 'Internet & Direct Marketing Retail'})
RETURN c.Symbol AS Symbol, c.Security AS Name, c.HeadquartersLocation AS Location
ORDER BY c.Symbol
"""
result = graph.query(industry_companies_query)
print("=== Internet & Direct Marketing Retail 산업 회사 ===")
for record in result:
    print(f"{record['Symbol']}: {record['Name']} ({record['Location']})")

print("\n✅ 데이터 확인 및 검색 완료")

=== 노드 타입별 개수 ===

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: Security)} {position: line: 4, column: 10, offset: 73} for query: "\nMATCH (c:Company {Symbol: 'AAPL'})\nRETURN c.Symbol AS Symbol, \n       c.Security AS Name,\n       c.Sector AS Sector,\n       c.SubIndustry AS Industry,\n       c.HeadquartersLocation AS Location,\n       c.Founded AS Founded\n"
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your q


Company: 502개
Exchange: 1개


=== Apple Inc. 정보 ===
Symbol: AAPL
Name: None
Sector: None
Industry: None
Location: None
Founded: None




Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: BELONGS_TO_SECTOR)} {position: line: 2, column: 21, offset: 21} for query: '\nMATCH (s:Sector)<-[:BELONGS_TO_SECTOR]-(c:Company)\nRETURN s.name AS Sector, count(c) AS CompanyCount\nORDER BY CompanyCount DESC\n'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you di

=== 섹터별 회사 수 ===


=== Information Technology 섹터 회사 (상위 10개) ===




Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: IN_INDUSTRY)} {position: line: 2, column: 21, offset: 21} for query: "\nMATCH (c:Company)-[:IN_INDUSTRY]->(i:Industry)-[:PART_OF_SECTOR]->(s:Sector)\nWHERE c.Symbol = 'AAPL'\nRETURN c.Security AS Company,\n       i.name AS Industry,\n       s.name AS Sector\n"
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query 

=== Apple의 관계 구조 ===


=== 거래소별 회사 수 ===
NYSE: 502개




Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: IN_INDUSTRY)} {position: line: 2, column: 21, offset: 21} for query: "\nMATCH (c:Company)-[:IN_INDUSTRY]->(i:Industry {name: 'Internet & Direct Marketing Retail'})\nRETURN c.Symbol AS Symbol, c.Security AS Name, c.HeadquartersLocation AS Location\nORDER BY c.Symbol\n"
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not 

=== Internet & Direct Marketing Retail 산업 회사 ===

✅ 데이터 확인 및 검색 완료


### 5. 벡터 인덱스 생성

In [85]:
# 여기에 코드를 작성하세요.

from langchain_openai import OpenAIEmbeddings
from langchain_neo4j import Neo4jVector

# OpenAI 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 1. Company 노드 데이터 조회
companies = graph.query("""
MATCH (c:Company)
RETURN c.Symbol AS Symbol,
       c.Security AS Security,
       c.Sector AS Sector,
       c.SubIndustry AS SubIndustry,
       c.HeadquartersLocation AS Location
""")

print(f"총 {len(companies)}개의 회사 데이터 조회 완료")

# 2. 각 Company에 대해 임베딩 생성 및 업데이트
from tqdm import tqdm

for company in tqdm(companies, desc="임베딩 생성 중"):
    # Company 정보를 텍스트로 결합
    combined_text = f"""회사명: {company['Security']}
심볼: {company['Symbol']}
섹터: {company['Sector']}
산업: {company['SubIndustry']}
본사 위치: {company['Location']}"""
    
    # 임베딩 생성
    embedding_vector = embeddings.embed_query(combined_text)
    
    # Company 노드에 임베딩 속성 추가
    graph.query("""
    MATCH (c:Company {Symbol: $symbol})
    CALL db.create.setNodeVectorProperty(c, 'embedding', $embedding)
    """, params={
        'symbol': company['Symbol'],
        'embedding': embedding_vector
    })

print("✅ 임베딩 생성 완료")

# 3. 벡터 인덱스 생성
graph.query("""
CREATE VECTOR INDEX company_vector_index IF NOT EXISTS
FOR (c:Company) ON (c.embedding)
OPTIONS {
  indexConfig: {
    `vector.dimensions`: 1536,
    `vector.similarity_function`: 'cosine'
  }
}
""")

print("✅ 벡터 인덱스 생성 완료")

# 4. 벡터 검색 테스트
vector_db = Neo4jVector.from_existing_index(
    embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name="company_vector_index",
    node_label="Company",
    text_node_property="Security",
    embedding_node_property="embedding",
        retrieval_query="""
    WITH node, score
    WITH node, score,
         "회사명: " + COALESCE(node.Security, "Unknown") + 
         "
심볼: " + COALESCE(node.Symbol, "N/A") +
         CASE WHEN node.Sector IS NOT NULL THEN "
섹터: " + node.Sector ELSE "" END +
         CASE WHEN node.SubIndustry IS NOT NULL THEN "
산업: " + node.SubIndustry ELSE "" END +
         CASE WHEN node.HeadquartersLocation IS NOT NULL THEN "
본사: " + node.HeadquartersLocation ELSE "" END
         AS combined_text
         
    RETURN combined_text AS text,
           score,
           {
               symbol: node.Symbol,
               security: node.Security,
               sector: node.Sector,
               subIndustry: node.SubIndustry,
               location: node.HeadquartersLocation
           } AS metadata
    """)

# 5. 유사도 검색 테스트
query = "technology companies in California"
results = vector_db.similarity_search(query, k=5)

print(f"\n검색 질의: '{query}'\n")
print("="*50)
for i, result in enumerate(results, 1):
    print(f"\n{i}. {result.page_content}")
    print(f"   메타데이터: {result.metadata}")

print("\n✅ 벡터 인덱스 생성 및 검색 테스트 완료")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: Security)} {position: line: 4, column: 10, offset: 55} for query: '\nMATCH (c:Company)\nRETURN c.Symbol AS Symbol,\n       c.Security AS Security,\n       c.Sector AS Sector,\n       c.SubIndustry AS SubIndustry,\n       c.HeadquartersLocation AS Location\n'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, ma

총 502개의 회사 데이터 조회 완료


임베딩 생성 중: 100%|██████████| 502/502 [04:19<00:00,  1.93it/s]


✅ 임베딩 생성 완료
✅ 벡터 인덱스 생성 완료


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: Security)} {position: line: 4, column: 34, offset: 226} for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \n    WITH node, score\n    WITH node, score,\n         "회사명: " + COALESCE(node.Security, "Unknown") + \n         "\n심볼: " + COALESCE(node.Symbol, "N/A") +\n         CASE WHEN node.Sector IS NOT NULL THEN "\n섹터: " + node.Sector ELSE "" END +\n         CASE WHEN node.SubIndustry IS NOT NULL THEN "\n산업: " + node.SubIndustry ELSE "" END +\n   


검색 질의: 'technology companies in California'


1. 회사명: Unknown
심볼: AAPL
   메타데이터: {'symbol': 'AAPL'}

2. 회사명: Unknown
심볼: TSLA
   메타데이터: {'symbol': 'TSLA'}

3. 회사명: Unknown
심볼: MSFT
   메타데이터: {'symbol': 'MSFT'}

4. 회사명: Unknown
심볼: CMCSA
   메타데이터: {'symbol': 'CMCSA'}

5. 회사명: Unknown
심볼: GOOGL
   메타데이터: {'symbol': 'GOOGL'}

✅ 벡터 인덱스 생성 및 검색 테스트 완료
